# Brent Oil Market AI — complete code and concepts guide

## A detailed, simple-English walkthrough of the whole project

This notebook is a **comprehensive learning companion** for the uploaded Oil Market News Analysis project. It keeps the full relevant project source code and explains each file, function, important statement, and concept in plain English. The notebook is organised into short sections so that the detail is easier to follow.

The project asks one narrow question:

> **Based on historical Brent oil prices, what probability does the model assign to Brent closing higher on the next trading day?**

It also collects recent oil-related news headlines and asks a language model (LLM) to score whether each headline is likely to push oil prices up or down. The two outputs are displayed side by side. **The current code does not train the price forecast on news scores.**

> **Research and learning only — not financial advice.** The included saved results are close to coin-flip accuracy, and a direction call is not a trading recommendation.


## How to use this notebook

1. Keep this notebook in the project root, next to `one_file.py` and the `oil_ai/` folder.
2. Read each explanation immediately before and after its code excerpt. Code cells reproduce the source code for study; they are not a sequence of standalone commands, have no saved outputs, and are not executed automatically.
3. Do **not** press **Run All** until the project environment is installed and you understand that some cells use the internet, news feeds, an LLM API, or write files.
4. The notebook never displays the private `.env` file. It only explains the safe `.env.example` template.
5. “Tomorrow” means the **next trading-day row in the data**, not always the next calendar day.

### What this notebook covers

- Python basics used by the project: imports, variables, functions, lists, dictionaries, loops, conditions, exceptions, modules and packages.
- Finance/data concepts: close price, return, volatility, moving average, volume, features, target and time order.
- Machine learning: logistic regression, scaling, regularisation, train/validation/test, probability, accuracy, Brier score and calibration.
- Engineering: RSS, HTTP, JSON, environment variables, SQLite, command-line flags, JSON output, charts and automated tests.
- Every Python source file in the uploaded project, including the test suite.

### Reading pattern used throughout

Each chapter introduces the purpose of a file, shows its original code in manageable sections, and follows it with an explanation of what the lines and Python ideas mean. When the same idea appears in several files, the later explanation connects it to the earlier one rather than assuming prior programming knowledge.

### Contents

1. Data and machine-learning foundations
2. Entry point and package map
3. Configuration, secrets, and price loading
4. The small teaching model
5. Experiments and model selection
6. The main price model and historical forecasts
7. News retrieval and LLM scoring
8. SQLite headline archive
9. Evaluation and charts
10. Reports and command-line interface
11. Tests, dependencies, outputs, limitations, and review questions


## Project flow in one picture

```text
.env settings (private) ─┐
                         ├─> config.py ─> cli.py (the coordinator)
Brent price CSV / Yahoo ─┘                    │
                                              ├─> model.py ─> price probability
Google News RSS ─> news.py ─> LLM scoring ────┤
                                              ├─> archive.py ─> SQLite database
                                              ├─> evaluate.py ─> metrics and PNG charts
                                              └─> report.py ─> terminal text / run.json

one_file.py is only the small entry door: it calls oil_ai.cli.main().
```

### Key rule: inputs must come before the answer

For a prediction made after the close on day **t**, the model may use information available on or before day **t**. Its target is whether the close on the **next** trading day, **t + 1**, is higher. If code uses the next-day close as an input, it has cheated by seeing the future. This is called **look-ahead bias** or **data leakage**.


## The real supplied data file

The included `data/brent_prices.csv` has the following verified structure:

| Item | Value |
|---|---:|
| Rows | 1,258 daily observations |
| Date range | 2021-10-14 to 2026-10-14 |
| Columns | `date`, `close`, `volume` |
| Missing values in those columns | 0 in the supplied file |

The first rows look like this:

```text
date        close       volume
2021-10-14  84.000000   37348.0
2021-10-15  84.860001   33044.0
2021-10-18  84.330002   40153.0
```

A **DataFrame** is a table like a spreadsheet. A **Series** is one labelled column from that table. The code uses the date as the row index so that rows stay in chronological order.


# 1. Concepts to understand before reading the code

## 1.1 Features and target

A **feature** is a number describing the market that was known when a prediction was made. Examples are yesterday’s return or the last ten days’ volatility. The model puts the features in a table called **X**.

The **target** is the answer the model is learning to predict. Here it is normally:

```text
1  if the next close is higher than today’s close
0  otherwise
```

The target is commonly stored as **y**. Keeping `X` (known information) separate from `y` (future outcome) is the central anti-cheating rule.

## 1.2 Return and volatility

- **One-day return:** `today_close / yesterday_close - 1`. A return of `0.02` means +2%.
- **Five-day return:** `today_close / close_five_trading_days_ago - 1`.
- **Volatility:** how much recent daily returns vary. It measures how jumpy the market has been, not whether the next move must be up or down.

## 1.3 Logistic regression and the sigmoid

Logistic regression adds a starting value (the **intercept**) and a weighted contribution from every feature. It then converts this score into a number between 0 and 1 with the **sigmoid** function:

```text
score       = intercept + (feature_1 × weight_1) + (feature_2 × weight_2) + ...
probability = 1 / (1 + e^(-score))
```

A score of zero becomes 50%. A positive score becomes more than 50%; a negative score becomes less than 50%. A model probability is not a guarantee, and an LLM news score is **not** a probability at all.

### What does “regression” mean here?

The name can be confusing. Logistic regression does not output a continuous oil price in this project. It is used as a **binary classifier**: it estimates how likely the target class `1` (“next close is higher”) is, then a threshold can turn that probability into a class call. Ordinary linear regression instead predicts a numeric quantity such as a price or return.

### How are the weights learned?

The weights are not chosen by hand and are not calculated from a simple rule like “positive returns always mean up.” During training the algorithm:

1. starts with candidate weights and an intercept;
2. computes a score and probability for each training example;
3. compares each probability with its known 0/1 target using **log loss**;
4. adjusts weights to make the total loss smaller; and
5. repeats until it finds a set of weights that fits the training data according to its optimisation procedure.

Log loss penalises a confident wrong prediction more strongly than a cautious one. Regularisation adds a penalty for very large weights. This encourages a simpler model and can reduce overfitting, although it cannot guarantee good performance on future market data.

### Tiny example of the learning problem

| Yesterday's return | Today's feature | Next day actually rose? |
|---:|---:|---:|
| +1% | 0.01 | 1 |
| -2% | -0.02 | 0 |
| +0.5% | 0.005 | 1 |

Each row is one training example. The feature is something known at prediction time; the final column is the answer the model learns from. With many examples, logistic regression chooses weights that assign higher probabilities to patterns associated with class 1 in the training data. It learns associations, not guaranteed causes.

### What does “regression” mean here?

The name can be confusing. Logistic regression does not output a continuous oil price in this project. It is used as a **binary classifier**: it estimates how likely the target class `1` (“next close is higher”) is, then a threshold can turn that probability into a class call. Ordinary linear regression instead predicts a numeric quantity such as a price or return.

### How are the weights learned?

The weights are not chosen by hand and are not calculated from a simple rule like “positive returns always mean up.” During training the algorithm:

1. starts with candidate weights and an intercept;
2. computes a score and probability for each training example;
3. compares each probability with its known 0/1 target using **log loss**;
4. adjusts weights to make the total loss smaller; and
5. repeats until it finds a set of weights that fits the training data according to its optimisation procedure.

Log loss penalises a confident wrong prediction more strongly than a cautious one. Regularisation adds a penalty for very large weights. This encourages a simpler model and can reduce overfitting, although it cannot guarantee good performance on future market data.

### Tiny example of the learning problem

| Yesterday's return | Today's feature | Next day actually rose? |
|---:|---:|---:|
| +1% | 0.01 | 1 |
| -2% | -0.02 | 0 |
| +0.5% | 0.005 | 1 |

Each row is one training example. The feature is something known at prediction time; the final column is the answer the model learns from. With many examples, logistic regression chooses weights that assign higher probabilities to patterns associated with class 1 in the training data. It learns associations, not guaranteed causes.

### What do the learned numbers mean?

- **Coefficient / weight:** change in the model's score associated with a one-unit change in that feature, holding the other features fixed. With standardised inputs, one unit means roughly one training standard deviation.
- **Intercept:** the model's starting score when all input values are zero (for standardised inputs, zero is approximately the training average).
- **Score/logit:** the weighted sum before the sigmoid.
- **Probability:** the sigmoid-transformed score, between 0 and 1.
- **Threshold:** a separate rule used to turn probability into a label. The project generally uses 0.5.

Mathematically, the score is also called the **log-odds**: `score = log(p / (1 - p))`, where `p` is the probability of class 1. Logistic regression is linear in this score/log-odds space; the sigmoid converts it back into a probability. For a one-unit feature increase, holding others fixed, the coefficient changes log-odds by that coefficient; the odds are multiplied by `exp(coefficient)`.

Weights do not directly say “importance” in every context. If features have different scales, their coefficients are not comparable by size. The main pipeline standardises first; `simple_model.py` does not.


In [ ]:
# Safe teaching example: weighted features -> score -> probability.
# This uses made-up weights and does not access project data or the internet.
import math

features = {"one_day_return": 0.01, "five_day_return": 0.02}
weights = {"one_day_return": 8.0, "five_day_return": 5.0}
intercept = -0.10

# Multiply each feature by its learned weight, then add the intercept.
score = intercept + sum(features[name] * weights[name] for name in features)
probability_of_up = 1 / (1 + math.exp(-score))
prediction = "UP" if probability_of_up > 0.5 else "NOT UP"

print(f"score = {score:+.3f}")
print(f"probability of an up day = {probability_of_up:.1%}")
print(f"class at the 50% threshold = {prediction}")


## 1.4 Chronological validation

This project aims to use three date-ordered blocks:

| Block | Job | Why it is separate |
|---|---|---|
| **Train** | Learn model weights from the oldest days | The model is allowed to learn from these days. |
| **Validation** | Compare alternative feature sets and regularisation values | Choose settings without touching the final test block. |
| **Test** | Grade the selected settings on newer unseen days | Gives the most honest final check. |

Randomly shuffling financial rows would blur the past/future boundary and can make results look stronger than they truly are.

## 1.5 Core quality metrics

- **Accuracy:** correct up/down calls divided by all graded calls. Around 50% is similar to a coin flip for a balanced directional problem.
- **Precision:** among the days the model called “up”, how many actually went up?
- **Recall:** among the days that actually went up, how many did the model identify?
- **Brier score:** average of `(predicted_probability − actual_outcome)²`. Lower is better; an always-50% forecast scores 0.25 on balanced binary outcomes.
- **Calibration/reliability:** if the model often says 60%, does the outcome occur roughly 60% of the time in that group?


### A simple Brier-score example

Suppose the model says 80% and the event happens (`actual = 1`). The squared error is `(0.8 - 1)² = 0.04`. If it says 20% and the event happens, the error is `(0.2 - 1)² = 0.64` — a confident miss receives a much larger penalty. Average this error over many predictions to get the Brier score. Lower is better.

### Why three data blocks?

The validation set answers “which configuration should we choose?” The test set answers “how did that already-chosen configuration perform on a later period?” If the test result is used repeatedly to change the model, it is no longer an independent final test. That repeated searching is often called **data snooping**.


# 2. Entry point and package map

The file you run is `one_file.py`. It does almost no work itself. It imports `main` from the `oil_ai` package and calls it only when the file is executed directly.


In [ ]:
# Original source: one_file.py (lines 1-18)
"""ONE-FILE OIL MARKET AI — thin compatibility shim.

All logic moved into the `oil_ai` package (config / prices / model / news /
archive / evaluate / report / cli). Running this file still launches the
exact same program as before:

    source .venv/bin/activate && python3 one_file.py

New flags since the modularization:
    --evaluate [N]     probability quality (Brier, calibration, precision/recall) + charts
    --archive-stats    summarize the SQLite news archive
    --json             write a machine-readable summary to results/run.json
"""

from oil_ai.cli import main

if __name__ == "__main__":
    main()


### Plain-English explanation

- Lines 1–13 are a **docstring**: text enclosed in triple quotes. It documents the file; Python stores it as documentation rather than executing it as program logic.
- Line 15 imports one function, `main`, from `oil_ai/cli.py`. The dot in `oil_ai.cli` means “the `cli.py` module inside the `oil_ai` package.”
- Line 17 uses the special variable `__name__`. It equals `"__main__"` only when this file is launched directly with `python3 one_file.py`.
- Line 18 starts the program. If another file imports `one_file.py`, this block does not run automatically.


The `__init__.py` file tells Python that `oil_ai` is a package. In this project it is documentation only; it does not define functions or run the pipeline.


In [ ]:
# Original source: oil_ai/__init__.py (lines 1-13)
"""oIL MARKET AI — package modules (ported from the original one_file.py).

Layout:
    config.py    shared settings: ticker, cache paths, drivers, AI prompt
    prices.py    download + cache Brent prices
    model.py     the auto-tuned logistic-regression model and its math
    news.py      fetch headlines + analyse each with an LLM
    archive.py   SQLite archive of analysed headlines (Stage 2 groundwork)
    evaluate.py  probability quality: accuracy, precision/recall, Brier,
                 reliability curve + charts
    report.py    all printed output: outlook, plain English, backtest
    cli.py       the command-line entry point (`one_file.py` calls this)
"""


### What the package layout achieves

Splitting the program into small modules makes the code easier to read and test. `config.py` owns settings; `prices.py` owns prices; `model.py` owns price forecasting; `news.py` owns headlines and LLM calls; `archive.py` owns SQLite storage; `evaluate.py` owns metrics and charts; `report.py` owns presentation; and `cli.py` decides what to run.


# 3. Configuration, secrets, and price loading

## 3.1 Shared constants and news-driver configuration (`oil_ai/config.py`)

This module holds values that several files need. Centralising constants reduces the risk that one path or market-driver label changes in one file but not another.


In [ ]:
# Original source: oil_ai/config.py (lines 1-31)
"""SHARED CONFIG — every constant the pipeline needs lives here, so there is
exactly one place to change a setting (the original one_file.py put these at
the top of the file)."""

import os
import urllib.parse

TICKER = "BZ=F"          # Brent crude front-month futures (Yahoo Finance)
PRICE_CSV = "data/brent_prices.csv"
RESULTS_DIR = "results"
ARCHIVE_DB = "data/news_archive.sqlite"   # analysed-headline archive (Stage 2)

# NOTE: PDF/PNG outputs from the evaluation module land in RESULTS_DIR.

DRIVERS = {   # the six market drivers the analyst focuses on (matches SYSTEM_PROMPT)
    "OPEC+ policy":        'OPEC+ OR "OPEC meeting" OR "production cuts" OR "production increase"',
    "Geopolitics":         'crude oil sanctions OR tanker OR "strait of hormuz" OR "oil supply disruption" OR war oil',
    "US inventories":     '"crude oil inventories" OR "EIA weekly petroleum" OR "API crude stocks"',
    "Dollar & macro":      '"US dollar" oil OR "Federal Reserve" oil OR "interest rates" oil OR "economic data" oil',
    "Price action":        'oil price rally OR crude oil selloff OR "oil prices surge" OR "oil prices slide"',
    "Market expectations": 'Brent outlook OR "oil price forecast" OR "oil demand outlook" OR traders expect',
}
DRIVER_ORDER = list(DRIVERS.keys())
RSS_BASE = "https://news.google.com/rss/search?q="


def rss_url(query):
    """Build a Google News RSS feed URL, filtered to the last 3 days so
    'past days + today' is covered."""
    return RSS_BASE + urllib.parse.quote_plus(query + " when:3d") + "&hl=en-US&gl=US&ceid=US:en"


### Plain-English explanation of lines 1–31

- Lines 1–3 document the purpose of the module.
- Lines 5–6 import standard-library modules. `os` reads environment variables later; `urllib.parse` safely encodes a web-search query.
- Lines 8–11 define constants. `BZ=F` is the Yahoo Finance symbol used for Brent futures data. The other strings are file or folder paths relative to the project root.
- Lines 15–22 define a **dictionary** named `DRIVERS`. Each dictionary key is a reader-friendly market-driver label; its value is a Google News search query. The six topics are intentionally broad: OPEC+ policy, geopolitics, inventories, macro, price action and market expectations.
- Line 23 converts the dictionary’s keys into a list. Python dictionaries preserve insertion order, so this list preserves the order in which the drivers were written.
- Line 24 is the first part of every Google News RSS search URL.
- Lines 27–30 define `rss_url(query)`. It appends `when:3d`, uses `quote_plus` to make the query safe for a URL, and adds language/country parameters. The `return` value is one complete RSS URL.


In [ ]:
# Original source: oil_ai/config.py (lines 33-58)
# =========================================================================
# CONFIG: which AI should analyse the news? (read from .env at run time)
# =========================================================================
SYSTEM_PROMPT = """You are an analyst specialising in crude oil markets.
You analyse news strictly through SIX market drivers:
  1. geopolitics      - wars, sanctions, attacks, shipping chokepoints, tanker risks
  2. opec_plus        - OPEC+ production policy, quota decisions, meetings
  3. inventories      - US crude oil inventory reports (EIA/API): builds are bearish, draws bullish
  4. macro            - US dollar strength (a stronger dollar pressures oil prices), interest rates,
                        Fed decisions, growth/recession data (incl. China demand)
  5. price_action     - oil's own moves: rallies, selloffs, momentum, breakouts
  6. expectations     - market-expectation changes: demand forecasts, analyst consensus, forward curves

Rules:
- Classify the headline into EXACTLY ONE driver: geopolitics, opec_plus, inventories, macro,
  price_action, or expectations (use 'other' only if it fits none).
- Do not assume a negative-sounding event is bearish for oil: judge its actual effect on
  supply, demand and market expectations (e.g. an output INCREASE is bearish; report of
  shipping-cost spikes from disruption is often bullish as a risk premium). - Distinguish confirmed facts from speculation; a headline may itself REPORT a market expectation,
- Do not invent numbers or probabilities.

Return ONLY a JSON object with fields:
event_summary, category (use a driver slug from the list above),
impact_direction (bullish/bearish/neutral), impact_score (-100..+100),
expected_horizon (intraday/24 hours/5 trading days), supply_impact, demand_impact,
uncertainty (low/medium/high), reasoning, key_assumption."""


In [ ]:
# Original source: oil_ai/config.py (lines 60-69)
DEFAULT_MODELS = {"openai": "gpt-4o-mini", "anthropic": "claude-sonnet-4-5", "gemini": "gemini-2.0-flash"}
OPENAI_BASE = {"openai": "https://api.openai.com/v1", "gemini": "https://generativelanguage.googleapis.com/v1beta/openai"}


def llm_config():
    """Resolve (provider, model, base_url) from the environment."""
    provider = os.getenv("LLM_PROVIDER", "openai").lower()
    model = os.getenv("LLM_MODEL") or DEFAULT_MODELS.get(provider, "gpt-4o-mini")
    base = os.getenv("LLM_BASE_URL") or OPENAI_BASE.get(provider, "http://localhost:11434/v1")
    return provider, model, base


### Plain-English explanation of lines 33–70

- Lines 36–58 assign a long multi-line string to `SYSTEM_PROMPT`. It is the instruction sent to the LLM before each headline. It asks for one category and a JSON-shaped answer with named fields. A prompt guides the LLM; it does not guarantee accuracy.
- Lines 60–61 are lookup dictionaries. `DEFAULT_MODELS` gives a fallback model name by provider, and `OPENAI_BASE` gives base URLs for compatible chat endpoints.
- Lines 64–69 define `llm_config()`. `os.getenv` reads a setting from the environment, with a default when it is missing. `.lower()` makes provider names case-insensitive. `or` means “use the left value if it is non-empty; otherwise use the right value.”
- The function returns a three-item tuple: `(provider, model, base_url)`. Other modules unpack it with `provider, default_model, base = llm_config()`.

### Safe configuration template

The project reads a private `.env` file at runtime. API keys must stay private, so the notebook intentionally shows only `.env.example`:


### Original source: `.env.example` (lines 1–23)

```ini
# Copy this file to ".env" and fill it in to enable the AI news analysis.
# The price-based prediction model works WITHOUT any API key.

# Which provider to use: openai | anthropic | gemini | custom (any OpenAI-compatible API)
LLM_PROVIDER=openai

# Your secret API key (get one from your provider's website)
LLM_API_KEY=

# The model name to use (optional; sensible defaults are used if left blank)
LLM_MODEL=

# Only needed for LLM_PROVIDER=custom (OpenAI-compatible base URL).
#
# Local option — run AI on your own computer with Ollama (https://ollama.com):
#   1. Install Ollama, then start it with:  ollama serve
#   2. Pull a model once with:              ollama pull gemma3:4b
#   3. Use these settings:
#      LLM_PROVIDER=custom
#      LLM_BASE_URL=http://localhost:11434/v1
#      LLM_API_KEY=ollama
#      LLM_MODEL=gemma3:4b
LLM_BASE_URL=
```


### Security note

`LLM_API_KEY` should never be placed in a notebook, committed to Git, or shared in screenshots. The price-only model can run without an API key; the key is needed only for LLM news analysis.


## 3.2 Price cache and Yahoo download (`oil_ai/prices.py`)

The function below first tries to reuse the local CSV cache. If the cache is missing or empty, it downloads five years of daily history and saves a simplified table.


In [ ]:
# Original source: oil_ai/prices.py (lines 1-24)
"""PRICES — download historical Brent crude prices and cache them in a CSV."""
import os

import pandas as pd
import yfinance

from .config import TICKER, PRICE_CSV


def load_prices():
    """Return daily Brent closes as a date-indexed DataFrame. Reads the CSV
    cache if it exists; otherwise downloads 5 years from Yahoo Finance and
    saves it for the next runs."""
    if os.path.exists(PRICE_CSV):
        prices = pd.read_csv(PRICE_CSV, index_col="date", parse_dates=["date"])
        if len(prices) > 0:
            return prices
    os.makedirs(os.path.dirname(PRICE_CSV), exist_ok=True)
    raw = yfinance.Ticker(TICKER).history(period="5y", interval="1d", auto_adjust=False).reset_index()
    prices = (pd.DataFrame({"date": pd.to_datetime(raw["Date"]).dt.tz_localize(None).dt.normalize(),
                            "close": raw["Close"].astype(float)})
              .dropna().drop_duplicates("date").sort_values("date").set_index("date"))
    prices.to_csv(PRICE_CSV)
    return prices


### Plain-English explanation of every part

- `import os`, `import pandas as pd`, and `import yfinance` make standard file tools, pandas tables, and Yahoo Finance access available. `as pd` is a common short name for pandas.
- `from .config ...` is a **relative import**. The leading dot means “from another module in this same package.”
- `load_prices()` is the one public function in this file.
- `os.path.exists(PRICE_CSV)` checks whether the cache file exists. If it does, `pd.read_csv(..., index_col="date", parse_dates=["date"])` reads it as a DataFrame, uses `date` as the row index and converts the dates from text to date/time values.
- `if len(prices) > 0` avoids returning an empty CSV.
- `os.makedirs(..., exist_ok=True)` creates `data/` when required. `exist_ok=True` prevents an error if the folder already exists.
- `yfinance.Ticker(TICKER).history(...)` makes the download request. `period="5y"` asks for five years, `interval="1d"` requests daily rows, and `auto_adjust=False` keeps unadjusted values.
- The long DataFrame expression on lines 20–22 creates only two columns: normalised non-time-zone `date` and floating-point `close`. It removes missing dates, duplicate dates, sorts date order and finally makes `date` the index.
- `prices.to_csv(PRICE_CSV)` persists the cache; the final `return` gives the caller the cleaned table.

**Important implementation detail:** the included cache contains `volume`, but the download code currently saves only `date` and `close`. A fresh download would therefore not contain a real volume column. In the rich feature set, the missing-volume fallback becomes a Series of zeros; `0 / rolling_mean(0) - 1` then becomes `0 / 0`, which is missing (`NaN`) for every row. Later `.dropna()` can remove every rich-feature row. Preserve/download real volume, or change the feature design, before relying on that fallback.


# 4. Start with the small teaching model (`simple_model.py`)

This is the best first file to read because it puts the complete price-only workflow in one place. It is separate from the more modular production-style pipeline.


In [ ]:
# Original source: simple_model.py (lines 1-34)
"""
THE SIMPLEST POSSIBLE OIL MODEL  (~30 essential lines)
======================================================

The whole idea in one sentence:
    "We give the model a list of examples: (what the oil market looked like
    each day) and (whether oil went up the next day). It learns how the
    two are connected."

The three ideas behind it, in simple words:

1. FEATURES (X)  -> simple numbers that describe the past,
                    like "how much did oil move yesterday?"
2. TARGET (y)    -> the answer we want to predict:
                    1 = oil went up the next day, 0 = it did not.
3. LOGISTIC REGRESSION -> gives each feature a WEIGHT (how important it is),
   adds the weighted numbers together, and turns that sum into a
   probability between 0 and 1 that oil goes up tomorrow.

We keep the data in date order: the model learns from the older days and is
tested on the newer days it has never seen.
(The common beginner mistake is to mix all days together. Then the model
accidentally sees future prices while learning, and its test score looks
much better than it really is. That mistake is called look-ahead bias.)

Run:  source .venv/bin/activate && python3 simple_model.py
"""

import math

import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score


### Lines 1–34 explained

- Lines 1–27 are an unusually detailed docstring. They introduce `X` for features, `y` for target, logistic regression, chronological testing and look-ahead bias.
- Line 29 imports `math`; it is intended for mathematical functions, although this particular file prints the sigmoid formula rather than calling `math` directly.
- Line 31 imports pandas as `pd` to work with tabular prices.
- Line 32 imports scikit-learn’s `LogisticRegression` estimator.
- Line 33 imports `accuracy_score`, a helper that compares predicted classes with actual classes.

An **estimator** is an object with learned state. Before `.fit(...)`, a logistic-regression object has settings but no learned weights. After `.fit(...)`, it has coefficients (`coef_`) and an intercept (`intercept_`).


In [ ]:
# Original source: simple_model.py (lines 35-59)
# ---- 1. Load the historical prices (already downloaded and saved) --------
prices = pd.read_csv("data/brent_prices.csv", index_col="date", parse_dates=["date"])
close = prices["close"]

# ---- 2. FEATURES: what the market looked like ON each day ---------------
X = pd.DataFrame({
    "yesterday_return": close.pct_change(),    # yesterday's move
    "week_return": close.pct_change(5),        # the last 5 trading days' move
    "volatility": close.pct_change().rolling(10).std(),  # how jumpy recent days are
})

# ---- 3. TARGET: is tomorrow's close above today's? -----------------------
y = (close.shift(-1) > close).astype(int)      # 1 = went up, 0 = went down or flat

data = pd.concat([X, y.rename("rose_next_day")], axis=1).dropna()
print(f"Learning examples (trading days): {len(data)}\n")

X, y = data[X.columns], data["rose_next_day"]

# ---- 4. Split in date order: learn from the past, test on the future ----
split = int(len(data) * 0.8)
X_tr, X_te = X.iloc[:split], X.iloc[split:]
y_tr, y_te = y.iloc[:split], y.iloc[split:]
print(f"First {split} days -> for LEARNING. Last {len(data) - split} days -> "
      f"for TESTING (the model never saw them while learning).\n")


### Lines 35–59 explained: load, create features, create target, split

- Line 36 reads the cached CSV. `index_col="date"` promotes the date column into the DataFrame index, while `parse_dates` converts text dates.
- Line 37 selects the `close` Series. A Series is one column with the same date index.
- Lines 40–44 construct a new DataFrame, `X`, from a dictionary of three named features. `pct_change()` means one-row percentage change; `pct_change(5)` means five-row percentage change; `.rolling(10).std()` is the ten-row rolling standard deviation of daily returns.
- Line 47 calls `.shift(-1)`: it lines the next row’s close up with today’s row. The comparison returns `True`/`False`; `.astype(int)` converts these to `1`/`0`.
- Line 49 joins the feature table and target column side-by-side, then `.dropna()` removes early rows that lack enough history for a five- or ten-day calculation.
- Line 50 uses an f-string. Text inside `{...}` is evaluated and inserted into the string.
- Line 52 reassigns `X` and `y` from the cleaned combined table. `X.columns` is the list of feature names.
- Line 55 calculates an 80% cutoff. `int(...)` removes the decimal part so it becomes a valid row position.
- Lines 56–57 use `.iloc`, which selects by row number rather than date label. `[:split]` means rows before the split; `[split:]` means rows from the split onward.
- Lines 58–59 print a human-readable reminder of the time-ordered split.


In [ ]:
# Original source: simple_model.py (lines 61-74)
# ---- 5. TRAIN: the model picks a weight for each feature ----------------
model = LogisticRegression(class_weight="balanced")
model.fit(X_tr, y_tr)

# ---- 6. TEST: accuracy on days the model has never seen -----------------
accuracy = accuracy_score(y_te, model.predict(X_te))
print(f"How often it gets the direction right on unseen future days: "
      f"{accuracy * 100:.1f}%  (50% would be a coin flip)\n")

# ---- 7. WHAT DID IT LEARN? ----------------------------------------------
print("The weight it picked for each feature "
      "(a bigger positive number = pushes 'goes up' harder):")
for name, weight in zip(X.columns, model.coef_[0]):
    print(f"  weight for {name:<18}: {weight:+.3f}")


### Lines 61–74 explained: train, test, inspect weights

- Line 62 makes a logistic-regression model. `class_weight="balanced"` increases the weight of whichever class is less common so that the model does not simply focus on the more common outcome.
- Line 63 fits the model using only the older training rows.
- Line 66 calls `.predict(X_te)`, which returns an up/down class for each test row. `accuracy_score` compares those classes against the held-out true labels.
- Lines 67–68 format the fraction as a percentage. `:.1f` means one decimal place.
- Lines 71–72 explain the interpretation of weights.
- Line 73 loops through paired column names and learned coefficients. `zip(...)` creates pairs, and `model.coef_[0]` accesses the one row of coefficients from a binary classifier.
- Line 74 prints each weight with a sign (`+` or `-`) and three decimal places.


In [ ]:
# Original source: simple_model.py (lines 76-90)
# ---- 8. TODAY'S FORECAST, WITH THE FULL CALCULATION SHOWN ---------------
today_values = X_te.iloc[-1]                    # yesterday's market picture
weights = model.coef_[0]
intercept = model.intercept_[0]                 # the starting score before features

print("\nTHE FULL CALCULATION for tomorrow (no hidden steps):")
print(f"  step 1: today's feature values (what the market looked like):")
for name, value in today_values.items():
    print(f"          {name:<18} = {value:+.4f}")

print(f"  step 2: multiply each feature value by its weight:")
pushes = {}
for name, value, weight in zip(X.columns, today_values, weights):
    pushes[name] = value * weight
    print(f"          {name:<18} ({value:+.4f}) x ({weight:+.3f}) = {pushes[name]:+.4f}")


In [ ]:
# Original source: simple_model.py (lines 92-106)
score = sum(pushes.values()) + intercept
print(f"  step 3: add all the pushes together, plus the starting score "
      f"({intercept:+.3f}):")
print(f"          total score = {score:+.4f}")

prob_up = model.predict_proba(X_te.iloc[[-1]])[0, 1]
print(f"  step 4: turn the score into a probability "
      f"(the formula is 1 / (1 + e^-score)):")
print(f"          probability = 1 / (1 + e^({score:+.4f})) = {prob_up * 100:.1f}%")

print(f"\nANSWER: Brent is predicted to be HIGHER on the next trading day "
      f"with {prob_up * 100:.0f}% probability "
      f"-> it leans {'UP' if prob_up > 0.5 else 'DOWN' if prob_up < 0.5 else 'NEITHER'}.")
print(f"Remember: 49% vs 51% is a very weak signal — near a coin flip. "
      f"Not financial advice.")


### Lines 76–106 explained: show one prediction step by step

- Line 77 selects the last test row as the market picture to explain. In this teaching script it is a held-out example, not necessarily today’s live forecast.
- Lines 78–79 retrieve the learned weights and intercept. The intercept is the score before any feature contribution is added.
- Lines 81–84 print every current feature value using `.items()`, which gives a `(name, value)` pair for each Series item.
- Line 87 creates an empty dictionary. Lines 88–90 loop through feature names, values and weights, multiply value × weight, store that result as a “push”, and print it.
- Line 92 adds every push plus the intercept to form a **linear score**.
- Lines 97–100 ask scikit-learn for the class probabilities. `[0, 1]` means first row, second class. For a binary classifier, the second class is `1` (the “up” outcome).
- Lines 102–104 print the final verbal direction. The conditional expression `A if condition else B` chooses text inline.
- Lines 105–106 state an essential limitation: a forecast barely above or below 50% is weak evidence.

**Learning warning:** unlike the main package pipeline, this compact script does not standardise feature scales. It is intentionally simple, but the production-style auto-tuned model uses `StandardScaler` in a pipeline.

**Important target detail:** line 47 makes the final row's missing next close compare as `False`; after conversion, that row can look like a known 0 (not-up). The final row should be excluded from training and testing because tomorrow's outcome is not known yet. The same pattern appears in other model-building code and is called out again there.


# 5. Fair configuration experiments (`experiments.py`)

This file is a small research lab. It tries basic and rich feature sets with three regularisation strengths, ranks them by validation accuracy, and only then looks at each leading configuration’s test score.


In [ ]:
# Original source: experiments.py (lines 1-31)
"""
EXPERIMENTS — can we honestly beat ~50% directional accuracy?
=============================================================

The golden rule this script enforces (this IS the interview lesson):
  TRAIN  -> the model learns here
  VALID  -> we COMPARE configurations here and pick the best
  TEST   -> used ONCE, at the end, only for the final pick.
If we tuned on the test set, we would fool ourselves into thinking we
improved — that's called data snooping, and it's why backtests lie.

Run:  python3 experiments.py        (uses the cached Brent CSV)
"""

import os

import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler


def load_brent_prices():
    """Read the price CSV cache (kept fresh by one_file.py). If it is missing,
    tell the user how to create it instead of failing with a cryptic error."""
    path = "data/brent_prices.csv"
    if not os.path.exists(path):
        raise RuntimeError("No price cache found. Run `python3 one_file.py --skip-news` "
                           "once to download Brent prices, then re-run experiments.")
    return pd.read_csv(path).dropna(subset=["close"])


### Lines 1–31 explained

- The docstring states the central discipline: train, then validate, then use the test data only for final grading.
- `os.path.exists(path)` checks that the cache exists before pandas tries to read it. Raising `RuntimeError` produces a clearer instruction than a low-level file-not-found failure.
- `pd.read_csv(path).dropna(subset=["close"])` reads the file and removes only rows missing a close price. `subset` says which column matters for this cleaning step.


In [ ]:
# Original source: experiments.py (lines 34-64)
# ---- feature builders: each returns X = "what the market looked like" -----
def features_basic(df, volume):
    """The 3 features the simple model used."""
    close = df["close"]
    return pd.DataFrame({
        "ret_1d": close.pct_change(),
        "ret_5d": close.pct_change(5),
        "volatility": close.pct_change().rolling(10).std(),
    }, index=df.index)


def features_rich(df, volume):
    """A wider view: several momentum horizons, ranges and trend distances."""
    close = df["close"]
    ret = close.pct_change()
    return pd.DataFrame({
        "ret_1d": ret, "ret_2d": close.pct_change(2), "ret_5d": close.pct_change(5),
        "ret_10d": close.pct_change(10), "ret_20d": close.pct_change(20),
        "vol_5d": ret.rolling(5).std(), "vol_10d": ret.rolling(10).std(),
        "vol_20d": ret.rolling(20).std(),
        "vs_ma20": close / close.rolling(20).mean() - 1,
        "vs_ma50": close / close.rolling(50).mean() - 1,
        "pos_in_range20": (close - close.rolling(20).min()) /
                          (close.rolling(20).max() - close.rolling(20).min()),
        "volume_z": volume / volume.rolling(20).mean() - 1,
    }, index=df.index)


FEATURE_SETS = {"basic": features_basic, "rich": features_rich}
REGULARIZATIONS = [0.001, 0.01, 0.1]  # smaller = stronger penalty = simpler model


### Lines 34–64 explained: two feature-builder functions and candidate settings

- Both functions take `df` (the price table) and `volume` (a Series). They return a feature DataFrame with the same row index as `df`.
- `features_basic` returns the three simple features: one-day return, five-day return and ten-day volatility.
- `features_rich` first creates `ret` so that repeated one-day-return calculations are not duplicated. It then returns 12 columns: momentum over several windows; three volatility windows; distance from 20/50-day moving averages; location within the recent 20-day high/low range; and relative volume versus its 20-day average.
- A moving-average distance is `close / rolling_mean − 1`. Positive means above the average.
- `pos_in_range20` is close to 0 near the recent low and close to 1 near the recent high. As with any division, a zero-width range would need care.
- `volume_z` is named like a z-score, but its implemented formula is **relative difference from the 20-day mean**, not the standard statistical z-score `(value − mean) / standard_deviation`.
- `FEATURE_SETS` maps labels to functions, allowing the later loop to call either builder.
- `REGULARIZATIONS` lists three values of `C`. In scikit-learn’s logistic regression, **smaller `C` means stronger regularisation** (more penalty/shrinkage), which generally produces a simpler, less flexible model.


In [ ]:
# Original source: experiments.py (lines 66-83)
def run():
    prices = load_brent_prices().dropna(subset=["close"])
    close = prices["close"]
    volume = prices["volume"] if "volume" in prices else pd.Series(0.0, index=prices.index)
    y = (close.shift(-1) > close).astype(int).rename("rose")

    results = []
    for set_name, builder in FEATURE_SETS.items():
        data = pd.concat([builder(prices, volume), y], axis=1).dropna()
        X, target = data.drop(columns="rose"), data["rose"]
        n = len(data)
        slices = {"train": (slice(0, n - 500)),
                  "valid": (slice(n - 500, n - 250)),
                  "test": (slice(n - 250, n))}
        X_tr, y_tr = X.iloc[slices["train"]], target.iloc[slices["train"]]
        X_va, y_va = X.iloc[slices["valid"]], target.iloc[slices["valid"]]
        X_te, y_te = X.iloc[slices["test"]], target.iloc[slices["test"]]


In [ ]:
# Original source: experiments.py (lines 84-109)
        for C in REGULARIZATIONS:
            pipe = make_pipeline(StandardScaler(),
                                 LogisticRegression(C=C, max_iter=5000, class_weight="balanced"))
            pipe.fit(X_tr, y_tr)
            valid_acc = accuracy_score(y_va, pipe.predict(X_va))
            # keep this config's OWN test slice so we never mix feature sets
            results.append((set_name, C, valid_acc, pipe, X_te, y_te))

    # ---- pick the top 3 on VALIDATION, score them ONCE on the untouched test set
    results.sort(key=lambda r: -r[2])
    baseline = max(y.iloc[: n - 250].mean(), 1 - y.iloc[: n - 250].mean())
    print(f"{'=' * 66}")
    print(f"  Leaderboard (validated on 250 unseen VALIDATION days, then tested ONCE\n"
          f"  on 250 untouched TEST days). Baseline (always most frequent): {baseline*100:.1f}%")
    print(f"{'=' * 66}")
    for rank, (set_name, C, valid_acc, pipe, X_te_cfg, y_te_cfg) in enumerate(results[:4], 1):
        test_acc = accuracy_score(y_te_cfg, pipe.predict(X_te_cfg))
        print(f"  {rank}. features={set_name:<6} C={C:<6} "
              f"validation {valid_acc*100:5.1f}%   test {test_acc*100:5.1f}%")
    print("\n  IMPORTANT: only the top-3's TEST scores are shown ONCE.")
    print("  If they cluster near validation, the finding is trustworthy;")
    print("  if they jump around, the 'improvement' was noise. No cherry-picking.")


if __name__ == "__main__":
    run()


### Lines 66–109 explained: the experiment loop and leaderboard

- `run()` is the procedure called when this file is launched directly.
- Lines 67–70 load prices, select the close and volume columns, and build the next-day target. The fallback `pd.Series(0.0, index=prices.index)` prevents a missing volume column from crashing the code.
- Line 72 starts an empty list to store each tested configuration.
- The first loop visits each feature set. Lines 74–76 construct the joined data, drop missing rolling-window rows, separate features from the target and count the usable rows.
- Lines 77–82 make chronological train/validation/test slices: all but the last 500 rows for training, the next 250 for validation, and the newest 250 for test.
- The nested loop tests each `C`. `make_pipeline(StandardScaler(), LogisticRegression(...))` guarantees the scaler is fitted only when the model is fitted. Scaling puts features on comparable units before regression weights are learned.
- `max_iter=5000` gives the optimiser many chances to converge. `class_weight="balanced"` compensates for different class frequencies.
- The tuple stored in `results` retains the validation accuracy, settings, trained pipeline and the matching test data for that same feature set.
- Lines 93–105 sort configurations by validation accuracy (descending with `-r[2]`), print a leaderboard, and calculate each displayed test accuracy after the validation ranking is decided.
- Lines 108–109 use the direct-execution pattern explained earlier.

**Caution about the printed baseline (line 94):** this does calculate a majority-class rate from the labels before the test block, but `n` is left over from the last feature-set loop. Basic and rich feature sets can have different usable row counts because rich features need more history, so the printed baseline is not aligned perfectly with each candidate’s data window. A fair comparison should compute a matching baseline for each candidate/window.

**Two more source details:** the target expression turns the final row’s unknown next-day result into 0 unless that row is explicitly removed. Also, although the comments say the top three test scores are displayed, the code uses `results[:4]`, so it prints four.


# 6. The modular price model (`oil_ai/model.py`)

This is the primary price-model module used by the command-line program. It contains splitting, feature construction, model building, automatic selection, an arithmetic display, and historical “time-travel” forecasting.


In [ ]:
# Original source: oil_ai/model.py (lines 1-56)
"""MODEL — auto-tuned logistic regression on "does Brent rise tomorrow?".

Instead of one fixed model, every run compares several honest candidates on
a validation window (never on the test window) and keeps the best.
The recipe, the features and the choice all live here. This is the port of
the embedded experiments logic from the original one_file.py.
"""

import math

import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

CANDIDATES = (0.001, 0.01, 0.1)   # regularization strengths to compare


def split_ranges(n, test_len=250, valid_len=250):
    """Chronological index slices: train -> validation -> test.
    Kept as its own (testable) function: the split must never overlap and
    must never let training data come after validation/test data."""
    return {
        "train": slice(0, n - test_len - valid_len),
        "valid": slice(n - test_len - valid_len, n - test_len),
        "test":  slice(n - test_len, n),
    }


def _features(kind, prices):
    """Basic = the original 3; Rich = the 12-feature momentum/volume set."""
    close = prices["close"]
    volume = prices["volume"] if "volume" in prices else pd.Series(0.0, index=prices.index)
    ret = close.pct_change()
    shared = {
        "ret_1d": ret, "ret_5d": close.pct_change(5),
        "vol_10d": ret.rolling(10).std(),
    }
    if kind == "rich":
        shared.update({
            "ret_2d": close.pct_change(2), "ret_10d": close.pct_change(10),
            "ret_20d": close.pct_change(20),
            "vol_5d": ret.rolling(5).std(), "vol_20d": ret.rolling(20).std(),
            "vs_ma20": close / close.rolling(20).mean() - 1,
            "vs_ma50": close / close.rolling(50).mean() - 1,
            "pos_in_range20": (close - close.rolling(20).min()) /
                              (close.rolling(20).max() - close.rolling(20).min()),
            "volume_z": volume / volume.rolling(20).mean() - 1,
        })
    return pd.DataFrame(shared, index=prices.index)


def _pipeline(C):
    return make_pipeline(StandardScaler(),
                         LogisticRegression(C=C, max_iter=5000, class_weight="balanced"))


### Lines 1–56 explained: imports, splits, features, and pipeline factory

- Lines 9–15 import mathematics, pandas, the logistic-regression estimator, accuracy metric, pipeline factory and standard scaler.
- Line 17 makes a tuple of three `C` values. A tuple is an ordered collection that is normally not changed.
- `split_ranges(n, test_len=250, valid_len=250)` returns a dictionary of Python `slice` objects. The training slice starts at zero and ends before validation; validation ends before test; test finishes at `n`. This preserves chronological order and makes the split easy to unit-test.
- `_features(kind, prices)` starts by taking `close` and then chooses `volume` if available, otherwise a zero-filled Series with the same dates. It makes `ret` once, then creates three shared basic features.
- When `kind == "rich"`, `.update({...})` adds nine more feature names and Series to the dictionary, yielding 12 features total. If `kind` is any other value, no update happens and the function returns only the basic three features.
- `pd.DataFrame(shared, index=prices.index)` assembles the dictionary of Series into a table, explicitly retaining the original date index.
- `_pipeline(C)` builds a two-step scikit-learn pipeline. The `StandardScaler` subtracts the training mean and divides by the training standard deviation; the logistic regression then works with those standardised values. A pipeline helps avoid accidentally fitting a scaler using test-period statistics.


In [ ]:
# Original source: oil_ai/model.py (lines 59-91)
def train_auto_model(prices, log=print):
    """Compare candidates on VALIDATION days, pick a winner, grade ONCE on the
    untouched TEST block, then refit on all history for today's forecast."""
    close = prices["close"]
    y_all = (close.shift(-1) > close).astype(int).rename("rose")

    results, test_sets = [], {}
    for kind in ("basic", "rich"):
        feats = _features(kind, prices)
        data = pd.concat([feats, y_all], axis=1).dropna()
        if len(data) < 700:
            continue
        X, t = data.drop(columns="rose").reset_index(drop=True), data["rose"].reset_index(drop=True)
        n = len(data)
        sl = split_ranges(n)
        X_tr, X_va, X_te = X.iloc[sl["train"]], X.iloc[sl["valid"]], X.iloc[sl["test"]]
        y_tr, y_va, y_te = t.iloc[sl["train"]], t.iloc[sl["valid"]], t.iloc[sl["test"]]
        for C in CANDIDATES:
            pipe = _pipeline(C)
            pipe.fit(X_tr, y_tr)
            results.append((accuracy_score(y_va, pipe.predict(X_va)), kind, C, pipe))
            test_sets[(kind, C)] = (X_te, y_te)

    if not results:
        raise RuntimeError("Not enough price history to train (need ~700 rows in the CSV).")
    results.sort(key=lambda r: -r[0])
    valid_acc, kind, C, winner = results[0]
    X_te, y_te = test_sets[(kind, C)]
    test_acc = accuracy_score(y_te, winner.predict(X_te))  # graded ONCE, like experiments.py
    baseline = max(valid_acc, 1 - valid_acc)
    log(f"Auto-tuned model: features={kind} C={C}   "
        f"validation {valid_acc*100:.1f}%   test {test_acc*100:.1f}%   "
        f"naive baseline ~{baseline*100:.1f}%")


In [ ]:
# Original source: oil_ai/model.py (lines 92-118)

    # Refit on ALL known history, then forecast the latest day (features are
    # all available today; the target (tomorrow's close) is what we predict).
    feats = _features(kind, prices)
    data = pd.concat([feats, y_all], axis=1).dropna()
    winner.fit(data[list(feats.columns)], data["rose"])
    prob = winner.predict_proba(feats.iloc[[-1]])[0, 1]

    # Keep every step of the arithmetic so the output can SHOW the calculation
    # (probability = sigmoid of the weighted feature sum for TODAY's market):
    clf = winner.named_steps["logisticregression"]
    scaler = winner.named_steps["standardscaler"]
    today = feats.iloc[-1]
    z = (today - scaler.mean_) / scaler.scale_     # standardized feature values
    pushes = z * clf.coef_[0]                      # each feature's push on the score
    logit = float(pushes.sum() + clf.intercept_[0])
    math_report = {"config": f"{kind} C={C}",
                   "features": dict(zip(feats.columns, [float(v) for v in today])),
                   "z": dict(zip(feats.columns, [float(v) for v in z])),
                   "weights": dict(zip(feats.columns, [float(v) for v in clf.coef_[0]])),
                   "pushes": dict(zip(feats.columns, [float(v) for v in pushes])),
                   "intercept": float(clf.intercept_[0]), "logit": logit}
    assert abs(float(1 / (1 + math.exp(-logit))) - prob) < 1e-6   # the math must match

    return {"prob": float(prob), "last": float(close.iloc[-1]),
            "date": prices.index[-1].date(), "accuracy": test_acc,
            "baseline": baseline, "config": f"{kind} C={C}", "math": math_report}


### Lines 59–118 explained: automatic selection, re-fit, prediction, and transparent math

- `train_auto_model(prices, log=print)` accepts the price table. The optional `log` parameter defaults to `print`, so a test could substitute another function to capture messages.
- Line 63 creates the next-day direction target. `rename("rose")` gives it a useful column name before it is joined to features.
- Lines 65–81 loop over `basic` and `rich` features, join features with the target, remove rows with missing rolling calculations, reset the index so that `.iloc` uses a simple 0…n−1 sequence, make chronological blocks and fit all three `C` candidates.
- Line 69 skips a feature set if fewer than roughly 700 usable rows remain, because the default 250/250 validation/test blocks would otherwise leave too little training data.
- Lines 77–79 train one pipeline and store its validation accuracy with its configuration.
- `test_sets[(kind, C)]` uses a tuple as a dictionary key, tying each test block to the exact feature set and regularisation value that produced it.
- Lines 82–91 check that at least one candidate exists, sort candidates by validation result, select the top one, then evaluate the selected pipeline on its untouched test block. `log(...)` prints a summary.
- Lines 93–98 rebuild the winning feature set, refit the already-created pipeline on the full joined data, and use the latest feature row to calculate an up probability.
- Lines 100–114 unpack the two pipeline stages. The code manually standardises today’s features, multiplies them by the learned weights, adds the intercept and calculates the sigmoid. This creates a transparent `math_report` dictionary containing every value needed for an explanation table.
- Line 114 is an `assert`: it stops the program if the independently calculated sigmoid and scikit-learn’s predicted probability differ by more than `1e-6`.
- Lines 116–118 return one dictionary for downstream reporting. It contains the probability, latest close/date, held-out test accuracy, baseline, configuration and full arithmetic.

### Important implementation caveats to understand

1. **Last-row label:** `(close.shift(-1) > close).astype(int)` turns the final row into `0` because the next close is missing and the comparison is false. `dropna()` does not remove it because that `0` is not missing. The final full-history re-fit at line 97 can therefore train on today’s features with an unknown tomorrow label treated as “down.” A safer construction would explicitly remove the last labelled row before fitting.
2. **Baseline:** line 88 defines `baseline = max(valid_acc, 1 - valid_acc)`. That is derived from the winning model’s validation accuracy, not from a simple independent baseline such as “always predict the more common class.” The label “naive baseline” is therefore misleading and should be corrected before using it for comparison.
3. **Forecast date:** the price data cache is not automatically refreshed when it already exists. “Latest” means latest row in the cached CSV, not necessarily today’s market close.


In [ ]:
# Original source: oil_ai/model.py (lines 121-140)
def show_forecast_math(report):
    """Print the forecast's full arithmetic, so no step stays hidden."""
    m = report["math"]
    print(f"{'-' * 62}\n")
    print(f"  HOW THE PRICE MODEL GOT ITS NUMBER — the full calculation ({m['config']})")
    print(f"{'-' * 62}")
    print("  The model gives each feature a weight. For every feature:")
    print("  today's value -> standardize it (how far from normal) -> x weight = push")
    print(f"\n  {'feature':<16} {'value':>9} {'standardized':>13} {'weight':>8} {'push':>9}")
    for col in m["features"]:
        print(f"  {col:<16} {m['features'][col]:>9.4f} {m['z'][col]:>13.2f} "
              f"{m['weights'][col]:>8.3f} {m['pushes'][col]:>9.4f}")
    print(f"\n  step 1: add all the pushes together, plus the starting score "
          f"({m['intercept']:+.4f}):")
    print(f"          score = {m['logit']:+.4f}")
    print(f"  step 2: turn the score into a probability: 1 / (1 + e^(-score))")
    print(f"          probability = 1 / (1 + e^({m['logit']:+.4f})) "
          f"= {report['prob'] * 100:.1f}%  "
          f"-> leans {'UP' if report['prob'] > 0.5 else 'DOWN' if report['prob'] < 0.5 else 'NEITHER'}")
    print(f"{'-' * 62}\n")


### Lines 121–140 explained: printing the arithmetic table

`show_forecast_math(report)` does not train or predict; it only displays the values saved in `math_report`.

- `m = report["math"]` is a shorter local name for the nested dictionary.
- The f-strings build a readable table. Alignment parts such as `<16`, `>9.4f` mean left/right aligned in a field of a certain width, with four decimal places for floats.
- The `for col in m["features"]` loop uses every feature name as a key to fetch the current value, z-score, weight and contribution.
- The final conditional expression determines whether the reported probability leans “UP”, “DOWN” or exactly “NEITHER.”

This output is valuable because it lets a reader verify the formula rather than treating the model as a black box.


In [ ]:
# Original source: oil_ai/model.py (lines 143-173)
# =========================================================================
# TIME TRAVEL: what WOULD the model have said on a past date?
# =========================================================================
# The golden rule for grading a dated prediction: on the forecast evening the
# model may only train on data whose OUTCOME was already known by then, and
# it may only see features of the chosen day. Never the day after. This is the
# same look-ahead prevention as the chronological split, applied to one date.


def forecast_on_date(prices, date_str):
    """Forecast made AS OF the chosen date (trains strictly on older data)."""
    ts = pd.Timestamp(date_str)
    loc = prices.index.searchsorted(ts, side="right") - 1   # last trading day on/before date
    if loc < 40:
        raise ValueError(f"Need ~40 trading days of history before {date_str} to train the model.")
    close = prices["close"].iloc[: loc + 1]
    features = pd.DataFrame({"yesterday_return": close.pct_change(),
                             "week_return": close.pct_change(5)})
    target = (close.shift(-1) > close).astype(int)
    data = pd.concat([features, target.rename("rose")], axis=1).dropna()
    usable = data.iloc[:-1]   # the final row's target would peek past the forecast date
    model = LogisticRegression(class_weight="balanced")
    model.fit(usable[["yesterday_return", "week_return"]], usable["rose"])
    chosen_day = prices.index[loc]
    prob = model.predict_proba(features.loc[[chosen_day]])[0, 1]
    actual = {"next_close": None, "next_date": None, "rose": None}
    if loc + 1 < len(prices):   # ⚠ only the OUTCOME fields read future data
        actual["next_close"] = prices["close"].iloc[loc + 1]
        actual["next_date"] = prices.index[loc + 1].date()
        actual["rose"] = actual["next_close"] > prices["close"].iloc[loc]
    return {"day": chosen_day.date(), "prob": prob, "close": float(close.iloc[-1]), **actual}


### Lines 143–174 explained: historical time-travel forecast

- The long comments explain the anti-look-ahead rule: a forecast dated `D` may train only on outcomes already known before or at that date.
- `pd.Timestamp(date_str)` converts the supplied text date to pandas’ timestamp type.
- `searchsorted(..., side="right") - 1` finds the last trading-day row on or before the requested calendar date. This handles weekends and holidays.
- The `loc < 40` guard prevents training with very little history.
- This historical function deliberately uses only two features: yesterday’s return and five-day return. It creates a target, joins it to the features, and uses `data.iloc[:-1]` to remove the last row because that row’s next-day outcome was unknown at the forecast point.
- A fresh `LogisticRegression(class_weight="balanced")` is fit on the usable historical rows. The classifier then predicts the chosen day.
- Lines 168–172 fill actual next-day outcome fields only after the prediction. The comments flag that those fields are for grading, not for training.
- The final dictionary uses `**actual` to merge the key/value pairs from the `actual` dictionary into the returned dictionary.

**Evaluation caveat:** this time-travel model is not the same as the main auto-tuned pipeline. It has two unscaled features, no validation-based selection and no `StandardScaler`. Therefore functions that call `forecast_on_date` evaluate this simple historical model, not the same rich-or-basic auto-tuned pipeline reported by `train_auto_model`.


# 7. News retrieval and LLM scoring (`oil_ai/news.py`)

The news component gathers RSS headlines under each of six driver topics, sends each headline to an LLM, parses the JSON-shaped response and combines scores. A positive score is meant to be “bullish” for oil; a negative score is “bearish.” It is an **impact rating**, not the chance that the price rises.


In [ ]:
# Original source: oil_ai/news.py (lines 1-47)
"""NEWS — fetch live headlines (one feed per driver) + analyse each with an LLM."""

import json
import os
from datetime import datetime
from statistics import variance

import requests
import xml.etree.ElementTree as ET

from .config import DRIVERS, rss_url, SYSTEM_PROMPT, llm_config


def llm_analysis(headline, model=None):
    """Send one headline to the configured LLM and get structured JSON back."""
    if not os.getenv("LLM_API_KEY"):
        print("! AI news analysis not configured — copy .env.example to .env (Ollama needs no real key).")
        return None
    provider, default_model, base = llm_config()
    model = model or default_model

    try:
        if provider == "anthropic":  # Anthropic has its own API shape.
            resp = requests.post("https://api.anthropic.com/v1/messages",
                headers={"x-api-key": os.environ["LLM_API_KEY"], "anthropic-version": "2023-06-01"},
                json={"model": model, "max_tokens": 1024, "temperature": 0.2, "system": SYSTEM_PROMPT,
                      "messages": [{"role": "user", "content": headline}]}, timeout=120)
        else:                        # OpenAI-compatible API (includes your local Ollama).
            resp = requests.post(f"{base.rstrip('/')}/chat/completions",
                headers={"Authorization": f"Bearer {os.environ['LLM_API_KEY']}"},
                json={"model": model, "temperature": 0.2,
                      "messages": [{"role": "system", "content": SYSTEM_PROMPT},
                                   {"role": "user", "content": headline}]}, timeout=120)
        resp.raise_for_status()
        answer = resp.json()["choices"][0]["message"]["content"] if provider != "anthropic" \
                 else "".join(b["text"] for b in resp.json()["content"])
        text = answer.strip().strip("`")
        text = text[4:] if text.lower().startswith("json") else text
        result = json.loads(text[text.find("{"): text.rfind("}") + 1])
        result["impact_score"] = max(-100, min(100, int(result.get("impact_score", 0))))
        return result
    except requests.exceptions.ConnectionError:
        print("! The AI server is not reachable — start Ollama (or run: ollama serve)")
        return None
    except Exception as err:
        print(f"! AI analysis failed for this headline ({type(err).__name__}: {err}) — it was skipped.")
        return None


### Lines 1–47 explained: LLM request and guarded JSON parsing

- Lines 3–9 import JSON parsing, environment-variable access, date parsing, sample variance, HTTP requests and XML tools.
- Lines 11 imports shared driver data, the RSS URL builder, the LLM prompt and the provider configuration function.
- `llm_analysis(headline, model=None)` is responsible for one headline. `model=None` means “use the default model unless the caller supplies one.”
- Lines 16–18 check whether `LLM_API_KEY` exists. If it does not, the function prints a setup message and returns `None`, which is Python’s “no usable result” value.
- Lines 19–20 obtain configured provider/model/base URL and prefer the caller’s chosen model when one is supplied.
- Lines 23–27 send Anthropic requests using its separate `/v1/messages` API shape and headers.
- Lines 29–33 send every other provider through an OpenAI-compatible chat endpoint. This supports OpenAI-compatible local servers such as Ollama if configured.
- `timeout=120` prevents the request from waiting forever. `raise_for_status()` turns HTTP error status codes into exceptions.
- Lines 35–36 extract the text from different JSON response shapes. The conditional expression chooses the OpenAI-compatible result unless the provider is Anthropic.
- Lines 37–40 trim whitespace and code fences, find the first `{` through last `}`, parse JSON and clamp `impact_score` to the integer range −100 through +100.
- The two `except` blocks handle connection failure separately and report any other error by type and message before returning `None`.

**Design limitation:** the parser assumes the LLM returns valid JSON with fields later used by reporting, such as `impact_direction`, `expected_horizon` and `reasoning`. The score is clamped, but the rest of the schema is not rigorously validated before use.


In [ ]:
# Original source: oil_ai/news.py (lines 50-69)
def fetch_headlines(total=10):
    """One dedicated feed PER DRIVER, so the mix covers all six focus areas.
    Every article is tagged with its driver at fetch time — more reliable
    than trusting the LLM's own category label."""
    per_driver = max(1, total // len(DRIVERS))
    articles = []
    for driver, query in DRIVERS.items():
        try:
            resp = requests.get(rss_url(query),
                                headers={"User-Agent": "Mozilla/5.0 (oil ai prototype)"}, timeout=20)
            resp.raise_for_status()
            items = list(ET.fromstring(resp.content).iter("item"))
        except Exception as err:
            print(f"! Driver feed '{driver}' failed: {err}")
            continue
        for item in items[:per_driver]:
            articles.append({"title": item.findtext("title", "").rsplit(" - ", 1)[0],
                             "published": item.findtext("pubDate", ""),
                             "driver": driver})
    return articles


In [ ]:
# Original source: oil_ai/news.py (lines 72-108)
def pub_day(pub):
    """'Tue, 13 Oct 2026 19:04:00 GMT' -> 'Tue 13 Oct' — which calendar day the
    news broke, so headlines from the past few days group into daily rows."""
    try:
        # RSS timestamps look like "Tue, 13 Oct 2026 19:04:00 GMT" — Python's
        # %z does not understand the letters "GMT", so swap them for +0000 first.
        zulu = pub.replace("GMT", "+0000").replace("UTC", "+0000").strip()
        # group by DAY only — headlines from the same calendar day share a row
        return datetime.strptime(zulu, "%a, %d %b %Y %H:%M:%S %z").strftime("%a %d %b")
    except ValueError:
        return pub or "pasted/unknown"


def aggregate(analyses):
    """The math behind the news tone: count directions, average the scores,
    and measure how far apart they are (sample variance)."""
    scores = [a["impact_score"] for a in analyses]
    n_bull = sum(a["impact_direction"] == "bullish" for a in analyses)
    n_bear = sum(a["impact_direction"] == "bearish" for a in analyses)
    avg = sum(scores) / len(scores) if scores else 0.0
    spread = variance(scores) if len(scores) > 1 else 0.0
    tone = "bullish" if avg > 5 else "bearish" if avg < -5 else "neutral"
    return n_bull, n_bear, avg, spread, tone


def drivers_table(analyses):
    """Per-driver scorecard: how hard is each of the six drivers pushing price?
    Headlines whose driver tag is unknown bucket into 'other'."""
    rows = []
    for driver in list(DRIVERS.keys()) + ["other"]:
        subset = [a for a in analyses
                  if (a.get("driver") if a.get("driver") in DRIVERS else "other") == driver]
        if not subset:
            continue
        n_bull, n_bear, avg, _spread, tone = aggregate(subset)
        rows.append((driver, len(subset), n_bull, n_bear, avg, tone))
    return rows


### Lines 50–108 explained: fetch RSS, parse dates, aggregate scores

- `fetch_headlines(total=10)` calculates `per_driver = max(1, total // len(DRIVERS))`. Integer floor division (`//`) rounds down. With six drivers, a request for 8 headlines yields one per driver, so it returns 6 rather than 8; this is a known behaviour of the code.
- The loop makes one RSS request per driver. The `User-Agent` identifies the program. If a feed fails, the exception is printed and the loop continues with other drivers.
- `ET.fromstring(resp.content).iter("item")` parses RSS XML and finds every `<item>` element.
- Each article dictionary stores the title (with a trailing “ - Publisher” removed), raw publication text and the driver tag chosen by the program. Tagging from the known feed avoids relying on the LLM to classify itself consistently.
- `pub_day(pub)` converts a typical RSS timestamp into a short day label. It converts `GMT`/`UTC` to `+0000` because the chosen parsing pattern expects a numeric time-zone offset. If parsing fails, it returns the original text or the fallback `pasted/unknown`.
- `aggregate(analyses)` extracts scores, counts exact `bullish`/`bearish` direction labels, calculates an arithmetic mean, calculates **sample variance** with `statistics.variance`, applies a ±5 neutral zone and returns five values as one tuple.
- `drivers_table(analyses)` groups results by each expected driver plus an `other` bucket. It returns rows of `(driver, count, bullish_count, bearish_count, average_score, tone)`.

A large reported `spread` is variance, not a standard deviation. Variance is in squared-score units, so a number such as 1,015 should not be read as “about 1,015 points apart.”


In [ ]:
# Original source: oil_ai/news.py (lines 111-143)
def day_trends(analyses):
    """Group the analysed news PER DAY, so we can see how the tone evolved."""
    by_day = {}
    for a in analyses:
        by_day.setdefault(a["day"], []).append(a)
    rows = []
    for day in sorted(by_day):
        n_bull, n_bear, avg, spread, tone = aggregate(by_day[day])
        rows.append((day, len(by_day[day]), n_bull, n_bear, avg, tone))
    return rows


def analyse_news(articles, model=None, save_to_archive=None):
    """Analyse every headline with the LLM. When `save_to_archive` is passed
    (the application database path from archive.py), each result is stored so
    a future news-fused model can use labelled history (Stage 2 groundwork)."""
    analyses = []
    for art in articles:
        print(f"\nHeadline: {art['title']}")
        print(f"Published: {art['published']}   (now's forecast may only use facts available already)")
        result = llm_analysis(art["title"], model=model)
        if result:
            result["day"] = pub_day(art["published"])     # which day the news belongs to
            result["driver"] = art.get("driver", "other")  # which market driver it hits
            result["headline"] = art["title"]
            analyses.append(result)
            print(f"  -> {result['impact_direction']:<7} score {result['impact_score']:+4.0f}"
                  f"  horizon: {result['expected_horizon']}")
            print(f"     {result['reasoning'][:180]}")
    if save_to_archive is not None and analyses:
        from . import archive
        archive.save(analyses, model, db_path=save_to_archive)
    return analyses


### Lines 111–143 explained: daily trend grouping and archive hand-off

- `day_trends(analyses)` builds a dictionary whose keys are day labels. `setdefault(key, [])` gets the existing list for a key or inserts an empty list first; `.append(a)` then adds an analysis to that day.
- It loops over `sorted(by_day)`, aggregates each day and returns summary rows. Because the keys are display strings such as `"Mon 12 Oct"`, this is alphabetical string sorting rather than robust date sorting. It works only loosely for the short feed window; storing a true date would be safer.
- `analyse_news(...)` loops through fetched article dictionaries, prints details, calls `llm_analysis`, adds the local day/driver/headline fields to successful results and appends them to `analyses`.
- `result["reasoning"][:180]` prints only the first 180 characters of the LLM’s explanation.
- If a database path was passed and at least one result exists, the function imports `archive` locally and calls `archive.save(...)`. The local import avoids making every use of `news.py` depend on archive setup at initial import time.
- The final `return analyses` gives callers the list of successful LLM analyses. Failed headlines are skipped rather than represented as empty records.

**Scope reminder:** the news scores are saved and displayed, but they do not enter the current logistic-regression features or probability calculation.

**Archival detail:** this function derives and saves `day`, but it does not copy `art["published"]` into `result`. Consequently, `archive.save` normally stores an empty `pub_raw` field even though the original raw publication text was available on the article.


# 8. SQLite headline archive (`oil_ai/archive.py`)

SQLite is a compact database stored in one file. It does not require a separate database server. This module creates a `headlines` table, inserts LLM analyses and produces simple grouped summaries.


In [ ]:
# Original source: oil_ai/archive.py (lines 1-43)
"""ARCHIVE — a tiny SQLite store for analysed headlines (Stage 2 groundwork).

Every run that analyses news saves the headline text, the day it was
published, the driver it was tagged with, and the AI's impact score. Over
time this becomes the labelled historical news dataset that the "missing
link" news-fused model (Model C) needs — without it, news can only ever be
shown side-by-side, never fused into training.

Schema (single table):
    headlines(id, saved_at, day, pub_raw, driver, title, direction,
              score, horizon, uncertainty, model)
"""

import os
import sqlite3
from datetime import datetime, timezone

from .config import ARCHIVE_DB

_SCHEMA = """
CREATE TABLE IF NOT EXISTS headlines (
    id          INTEGER PRIMARY KEY AUTOINCREMENT,
    saved_at    TEXT NOT NULL,
    day         TEXT NOT NULL,      -- grouped calendar day, e.g. 'Tue 13 Oct'
    pub_raw     TEXT,
    driver      TEXT NOT NULL,
    title       TEXT NOT NULL,
    direction   TEXT,
    score       INTEGER,
    horizon     TEXT,
    uncertainty TEXT,
    model       TEXT
);
"""


def connect(db_path=ARCHIVE_DB):
    """Open (and create) the archive database with its schema."""
    os.makedirs(os.path.dirname(db_path), exist_ok=True)
    conn = sqlite3.connect(db_path)
    conn.executescript(_SCHEMA)
    return conn


### Lines 1–43 explained: table schema and connection

- The docstring explains the long-term goal: accumulate historical scored headlines so a later project version could evaluate whether news improves prediction.
- `sqlite3` is in Python’s standard library. `datetime, timezone` are used to stamp records in UTC.
- `_SCHEMA` is a multi-line SQL string. `CREATE TABLE IF NOT EXISTS` creates the table only if it is not already present. Each column declares a name and type; `id INTEGER PRIMARY KEY AUTOINCREMENT` gives each row a new unique numeric identifier.
- `connect(db_path=ARCHIVE_DB)` creates the parent folder if necessary, opens/creates the SQLite file, runs the schema script, and returns the open connection. The caller must later close that connection.


In [ ]:
# Original source: oil_ai/archive.py (lines 45-62)
def save(analyses, model=None, db_path=ARCHIVE_DB):
    """Store one run's analysed headlines. Returns the number of rows saved."""
    if not analyses:
        return 0
    now = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S")
    rows = [(now, a.get("day", ""), a.get("published", ""), a.get("driver", "other"),
             a.get("headline", ""), a.get("impact_direction", ""),
             int(a.get("impact_score", 0)), str(a.get("expected_horizon", "")),
             str(a.get("uncertainty", "")), model or "") for a in analyses]
    conn = connect(db_path)
    try:
        with conn:
            conn.executemany(
                "INSERT INTO headlines(saved_at, day, pub_raw, driver, title, direction,"
                " score, horizon, uncertainty, model) VALUES (?,?,?,?,?,?,?,?,?,?)", rows)
    finally:
        conn.close()
    return len(rows)


In [ ]:
# Original source: oil_ai/archive.py (lines 65-97)
def stats(db_path=ARCHIVE_DB, limit_days=30):
    """Two useful summaries for the archive: score per published day and
    score per driver. Returns (rows_by_day, rows_by_driver)."""
    if not os.path.exists(db_path):
        return [], []
    conn = connect(db_path)
    try:
        by_day = conn.execute(
            "SELECT day, COUNT(*), AVG(score) FROM headlines"
            " GROUP BY day ORDER BY COUNT(*) DESC, day LIMIT ?", (limit_days,)).fetchall()
        by_driver = conn.execute(
            "SELECT driver, COUNT(*), AVG(score) FROM headlines"
            " GROUP BY driver ORDER BY ABS(AVG(score)) DESC").fetchall()
    finally:
        conn.close()
    return by_day, by_driver


def print_stats(db_path=ARCHIVE_DB):
    """CLI-friendly printout of what the archive has collected so far."""
    by_day, by_driver = stats(db_path)
    if not by_day:
        print("The news archive is empty — run the program once with news analysis.")
        return
    print(f"{'=' * 62}\n  NEWS ARCHIVE — labelled history for the future news-fused model")
    print(f"{'=' * 62}")
    total = sum(n for _d, n, _s in by_day)
    print(f"  {total} saved headlines. Average score per published day:")
    for day, n, avg in by_day[:10]:
        print(f"   {day:<14} {n:>3} headlines   average score {avg:+7.1f}")
    print("\n  Average score per driver so far:")
    for driver, n, avg in by_driver:
        print(f"   {driver:<22} {n:>3} headlines   average score {avg:+7.1f}")


### Lines 45–98 explained: save, query and print archive statistics

- `save(analyses, model=None, db_path=ARCHIVE_DB)` immediately returns `0` when the supplied list is empty, avoiding creation of a database with no rows.
- Line 49 generates an ISO-like UTC save time once for the batch.
- Lines 50–53 build one tuple per analysis. `.get(key, default)` protects against missing dictionary keys. `int(...)` turns the score into an integer; `str(...)` makes optional values safe to store as text.
- `conn.executemany(SQL, rows)` runs the parameterised `INSERT` statement once for every tuple. The `?` placeholders are important: they pass values safely instead of building SQL by string concatenation.
- `with conn:` commits successful work (or rolls it back if an error occurs); `finally: conn.close()` closes the connection even when an error is raised.
- `stats(...)` returns two query result lists. SQL `GROUP BY` groups records; `COUNT(*)` counts headlines; `AVG(score)` calculates mean score. `LIMIT ?` receives a safely parameterised row limit.
- `print_stats(...)` is a terminal display function. It handles an empty archive, calculates total stored records, then loops through the query results to print formatted summaries.

**Data-quality limitations:** the schema has no uniqueness rule for a headline, so the same story fetched repeatedly can be stored repeatedly and distort later averages unless deduplication is added. Also, `limit_days=30` limits the query to 30 *textual day groups ordered by headline count*, not to the latest 30 calendar days. When there are more groups than the limit, the displayed total is not a complete all-time archive total.


# 9. Probability evaluation and chart creation (`oil_ai/evaluate.py`)

Accuracy is useful, but a probability forecast also needs to be checked for calibration. This module walks through recent historical dates, calls the time-travel forecast function, calculates metrics, prints a report and saves charts.


In [ ]:
# Original source: oil_ai/evaluate.py (lines 1-46)
"""EVALUATE — probability quality, not just accuracy.

The original program only reported directional accuracy (~50% = a coin
flip). An honest probability model must also be asked: are the numbers
themselves trustworthy? This module walks the model forward over recent
history, then computes:

  - accuracy          share of days the >50% rule called the direction right
  - precision         of days we called UP, how many actually went up
  - recall            of days that actually went up, how many we called
  - Brier score       mean (probability - outcome)^2; 0 = perfect, 0.25 = always guessing 50%
  - reliability bins  grouped by predicted probability: does a "60%" really rise 60% of the time?

and saves two charts into results/: a reliability curve with the forecasts
over time, and the model's learned feature weights.

Run via the CLI:  python3 one_file.py --evaluate 30
"""

import os

import matplotlib
matplotlib.use("Agg")            # no display server needed — we only save files
import matplotlib.pyplot as plt

from .config import RESULTS_DIR
from .model import forecast_on_date


def collect_forecasts(prices, n_days=60):
    """Walk the model forward over the last n_days trading days and collect
    (date, predicted_probability, actually_rose, next_close). Training for
    each day uses ONLY data from before that day (no look-ahead)."""
    n_days = max(1, min(int(n_days), len(prices) - 46))
    rows = []
    for loc in range(len(prices) - n_days, len(prices)):
        day = prices.index[loc].date()
        try:
            info = forecast_on_date(prices, str(day))
        except ValueError:
            continue
        if info["rose"] is None:      # most recent day may have no outcome yet
            continue
        rows.append((day, float(info["prob"]), int(bool(info["rose"])),
                     float(info["next_close"])))
    return rows


### Lines 1–46 explained: non-interactive plotting and walk-forward rows

- Lines 1–18 are a module docstring listing the evaluation measures and outputs.
- `matplotlib.use("Agg")` is called before importing `pyplot`. `Agg` is a non-graphical backend: it lets the program save PNG files even on a server with no display.
- `collect_forecasts(prices, n_days=60)` bounds the requested count. `max(1, min(...))` keeps it at least one and no more than the data can support while allowing about 46 rows for look-back history.
- The `for loc in range(...)` loop visits each selected price row in chronological sequence.
- For each day, `forecast_on_date` is called with that day. A `ValueError` means insufficient history, so that date is skipped.
- If the returned outcome is still `None`, the day has no future close in the available data and cannot be graded.
- Otherwise, a tuple of date, probability, 0/1 outcome and next close is appended to `rows`.

Because it calls `forecast_on_date`, these rows evaluate the simpler two-feature historical model described in the earlier caveat, not the auto-tuned scaled model used for the latest headline report.


In [ ]:
# Original source: oil_ai/evaluate.py (lines 49-87)
def metrics(rows):
    """Accuracy / precision / recall / Brier score for collect_forecasts rows."""
    n = len(rows)
    if n == 0:
        return {"n": 0, "accuracy": None, "precision": None,
                "recall": None, "brier": None, "mean_prob": None}
    probs = [r[1] for r in rows]
    truth = [r[2] for r in rows]
    preds = [int(p > 0.5) for p in probs]   # int on purpose: False == 0 is True in Python
    tp = sum(1 for p, t in zip(preds, truth) if p and t)
    called_up = sum(1 for p in preds if p)
    actual_up = sum(1 for t in truth if t)
    return {
        "n": n,
        "accuracy": sum(1 for p, t in zip(preds, truth) if p == t) / n,
        "precision": tp / called_up if called_up else None,
        "recall": tp / actual_up if actual_up else None,
        "brier": sum((p - t) ** 2 for p, t in zip(probs, truth)) / n,
        "mean_prob": sum(probs) / n,
    }


def reliability_bins(rows, n_bins=5):
    """Group forecasts by predicted probability and compare each group's
    mean prediction with its observed up-rate. Trustworthy = the two match."""
    bins = []
    for i in range(n_bins):
        lo, hi = i / n_bins, (i + 1) / n_bins
        pts = [r for r in rows if lo <= r[1] < hi]
        if i == n_bins - 1:   # include an exact-1.0 prediction in the last bin
            pts += [r for r in rows if r[1] == hi]
        if pts:
            bins.append({"lo": lo, "hi": hi, "n": len(pts),
                         "mean_pred": sum(r[1] for r in pts) / len(pts),
                         "mean_actual": sum(r[2] for r in pts) / len(pts)})
        else:
            bins.append({"lo": lo, "hi": hi, "n": 0,
                         "mean_pred": (lo + hi) / 2, "mean_actual": None})
    return bins


### Lines 49–87 explained: metrics and reliability bins

- `metrics(rows)` handles an empty list explicitly, returning a consistent dictionary whose unavailable metrics are `None`.
- `probs` and `truth` use list comprehensions to pull the probability and actual-outcome positions from every tuple.
- `preds = [int(p > 0.5) for p in probs]` applies a strict threshold: exactly 50% counts as a non-up call (`0`).
- `tp` means true positives: the number of rows where the model called up and the market went up. `called_up` and `actual_up` are denominator counts.
- Accuracy compares every predicted class with truth. Precision and recall use guards so the code does not divide by zero. The Brier score averages squared probability errors.
- `reliability_bins(rows, n_bins=5)` makes equal-width probability bins. For five bins, they are 0–20%, 20–40%, and so on.
- `pts = [...]` selects rows whose probability falls inside one bin. The last-bin exception includes an exact 1.0 prediction, which would otherwise be excluded by `< hi`.
- Non-empty bins get mean predicted probability and observed up rate; empty bins keep `n=0` and `mean_actual=None`.


In [ ]:
# Original source: oil_ai/evaluate.py (lines 90-120)
def print_report(rows, m, log=print):
    """Human-readable summary of the evaluation."""
    print(f"\n{'=' * 62}")
    print("  PROBABILITY EVALUATION — quality of the price model's own numbers")
    print(f"{'=' * 62}")
    if m["n"] == 0:
        log("  Not enough graded days to evaluate. (Needs a few days of newer prices.)")
        return
    log(f"  Graded days:            {m['n']}")
    log(f"  Directional accuracy:   {m['accuracy'] * 100:5.1f}%   (50% = coin flip)")
    if m["precision"] is not None:
        log(f"  Precision (called UP):  {m['precision'] * 100:5.1f}%   of the days we said UP, how many rose")
    if m["recall"] is not None:
        log(f"  Recall (real UP days):  {m['recall'] * 100:5.1f}%   of all up-days, how many we caught")
    if m["brier"] is not None:
        quality = ("excellent" if m["brier"] < 0.2 else
                   "decent" if m["brier"] < 0.24 else
                   "no better than always guessing 50%" if abs(m["brier"] - 0.25) < 0.01 else
                   "poor")
        log(f"  Brier score:            {m['brier']:.4f}   (0 = perfect, 0.25 = meaningless) -> {quality}")
    if m["mean_prob"] is not None:
        drift = (m["mean_prob"] - .5) * 100
        log(f"  Mean predicted prob:    {m['mean_prob'] * 100:5.1f}%   "
            f"({'biased ' + f'{drift:+.0f}pp toward UP' if abs(drift) > 2 else 'roughly unbiased'})")
    print("  RELIABILITY (does a stated 60% actually rise ~60% of the time?):")
    for b in reliability_bins(rows):
        if b["n"] == 0:
            continue
        print(f"   bins {b['lo']:.0%}-{b['hi']:.0%}: {b['n']:>3} forecasts   "
              f"predicted {b['mean_pred']:5.1%}   actually rose {b['mean_actual']:5.1%}")


### Lines 90–120 explained: readable evaluation report

- `print_report(rows, m, log=print)` prints a header. It uses `log` for most messages so a caller can replace it in tests or another interface.
- `m["n"] == 0` gives a friendly early exit when nothing can be graded.
- Lines 98–113 format counts, accuracy, precision, recall, Brier score and mean probability.
- The nested conditional expression on lines 105–108 gives a plain-English Brier interpretation. It is only a simple label, not a formal statistical test.
- `drift = (mean_prob - .5) * 100` measures the average probability’s distance from an even 50/50 forecast in percentage points.
- The loop over `reliability_bins(rows)` prints only populated bins.


In [ ]:
# Original source: oil_ai/evaluate.py (lines 122-153)
def plot_evaluation(rows, math_report=None, out_dir=RESULTS_DIR):
    """Save two PNG charts into results/: (1) reliability + hit-rate over
    time, (2) the currently learned feature weights."""
    os.makedirs(out_dir, exist_ok=True)

    # Chart 1 — reliability curve (left) and running hit-rate (right)
    fig, (ax_rel, ax_run) = plt.subplots(1, 2, figsize=(11, 4.2))
    bins = [b for b in reliability_bins(rows) if b["n"] > 0]
    if bins:
        ax_rel.plot([b["mean_pred"] for b in bins], [b["mean_actual"] for b in bins],
                    "o-", label="observed up-rate")
        ax_rel.plot([0, 1], [0, 1], "k:", label="perfect calibration")
        ax_rel.set_xlim(0, 1); ax_rel.set_ylim(0, 1)
        ax_rel.set_xlabel("predicted probability of rise")
        ax_rel.set_ylabel("observed share that rose")
        ax_rel.set_title("Reliability curve (calibration)")
        ax_rel.legend(); ax_rel.grid(alpha=.3)
    if rows:
        days = [r[0] for r in rows]
        runs, hits = [], 0
        for r in rows:
            hits += (r[1] > 0.5) == bool(r[2])
            runs.append(hits)
        ax_run.plot(days, runs, "o-")
        ax_run.set_title("Correct calls over time (cumulative)")
        ax_run.set_ylabel("days graded correct"); ax_run.grid(alpha=.3)
        fig.autofmt_xdate()
    fig.suptitle("Oil Market AI — probability evaluation (walk-forward, no look-ahead)")
    fig.tight_layout()
    path1 = os.path.join(out_dir, "evaluation.png")
    fig.savefig(path1, dpi=150); plt.close(fig)


In [ ]:
# Original source: oil_ai/evaluate.py (lines 154-169)
    # Chart 2 —feature weights of today's model
    path2 = None
    if math_report:
        w = math_report["weights"]
        order = sorted(w, key=lambda k: w[k])
        fig2, ax = plt.subplots(figsize=(7, max(3, .35 * len(order) + 1.5)))
        ax.barh(order, [w[k] for k in order])
        ax.axvline(0, color="black", lw=.8)
        ax.set_title("Learned feature weights (rich feature set)")
        ax.set_xlabel("weight on the up-probability")
        fig2.tight_layout()
        path2 = os.path.join(out_dir, "feature_weights.png")
        fig2.savefig(path2, dpi=150); plt.close(fig2)

    print(f"\n  Charts saved: {path1}" + (f" and {path2}" if path2 else ""))
    return path1, path2


### Lines 122–169 explained: save the PNG charts

- `plot_evaluation(...)` makes the result folder and builds a figure with two axes: reliability on the left and running correct-call count on the right.
- `bins = [...]` filters away empty bins. The reliability curve plots mean prediction against actual up rate; the dotted diagonal is perfect calibration.
- The running chart walks through rows, tests whether `(probability > .5)` matches the actual Boolean outcome, increments a cumulative `hits` count and plots that count against dates.
- `fig.tight_layout()` reduces overlapping labels. `fig.savefig(...)` writes a PNG, and `plt.close(fig)` releases the figure from memory.
- When `math_report` exists, the second chart sorts current feature weights and draws a horizontal bar chart. It is labelled “rich feature set” even if the winning configuration is basic; that title is therefore potentially inaccurate.
- The function prints paths and returns `(path1, path2)`.


In [ ]:
# Original source: oil_ai/evaluate.py (lines 172-184)
def evaluate_run(prices, math_report=None, n_days=60, log=print):
    """Full evaluation pass: collect walk-forward forecasts, print the report,
    save the charts, and return a machine-readable summary (for --json)."""
    rows = collect_forecasts(prices, n_days)
    m = metrics(rows)
    print_report(rows, m, log=log)
    paths = plot_evaluation(rows, math_report)
    m["results_png"] = paths[0]
    if paths[1]:
        m["weights_png"] = paths[1]
    if rows:
        m["reliability_bins"] = reliability_bins(rows)
    return m


### Lines 172–184 explained: the one-call evaluation wrapper

`evaluate_run(...)` is a convenience function. It collects walk-forward rows, computes metrics, prints the report, saves charts, adds chart paths into the metrics dictionary, adds reliability-bin details when rows exist, and returns that dictionary. The `--json` command-line flag later stores this returned summary in `results/run.json`.


# 10. Presentation functions (`oil_ai/report.py`)

This module should not change the model or scrape data. Its role is to turn pre-computed dictionaries/lists into human-readable terminal output. Keeping presentation separate from calculations makes debugging easier.


In [ ]:
# Original source: oil_ai/report.py (lines 1-48)
"""REPORT — everything the program prints: the outlook table, the
plain-English verdict, the backtest, the time-travel report and the
one-paragraph takeaway at the very end."""

from .model import forecast_on_date
from .news import aggregate, drivers_table, day_trends


def outlook(prob, last, date, accuracy, analyses, model_name, baseline=None):
    ag = aggregate(analyses)
    n_bull, n_bear, avg, spread, tone = ag
    down_or_up = "more likely DOWN" if prob < 0.5 else "more likely UP"
    base_line = f"   (a naive baseline to beat: {baseline * 100:.1f}%)" if baseline else ""
    print(f"""
{'=' * 62}
  OIL MARKET AI  (AI model: {model_name})
{'=' * 62}
  Brent closing price:             ${last:,.2f} ({date})
     -> the oil price this whole program works from.
  Accuracy on days it never saw:   {accuracy * 100:.1f}%{base_line}
     -> how often the model's UP/DOWN call was right on unseen days; 50% = random.
  Chance of going up (price only): {prob * 100:.0f}%  -> {down_or_up}
     -> this comes straight from the weighted-feature math shown above.

  Headlines analysed:              {len(analyses)}
     -> each headline was given a score from -100 (pushes oil down) to
        +100 (pushes oil up) by the AI.
  Pushing up / pushing down:       {n_bull} / {n_bear} headlines
     -> headlines whose score leaned positive / negative.
  Average impact score:            {avg:+.1f} (-100 = very bearish, +100 = very bullish)
     -> the mean of every headline score: the overall news push.
  How much headlines disagree:     {spread:.0f}  (higher = a noisier news day)
     -> how far apart the individual scores are.
  Overall news tone:               {tone}
     -> bullish if the average is above +5, bearish below -5, else neutral.

  BREAKDOWN BY MARKET DRIVER (one row per theme: OPEC+, geopolitics, etc.):
""")
    for driver, n, bull, bear, davg, dtone in drivers_table(analyses):
        print(f"   {driver:<20} {n:>2} headlines   {bull} bull / {bear} bear   "
              f"net {davg:+6.1f}   ({dtone})")
    print("""
     bull/bear = headlines pushing oil UP / DOWN; net = their average score.
  NOTE: the probability is trained on price features only (Model A); news is
  reported separately until the news archive has enough labelled history
  (Stage 4) to retrain with news included (Model C). Research tool, NOT
  financial advice.
""")


### Lines 1–48 explained: the main outlook table

- Lines 5–6 import the forecast function and the three news-summary helpers.
- `outlook(...)` receives already-calculated model probability, latest close/date, held-out accuracy, news analyses, model name and an optional baseline.
- Lines 10–11 unpack the five values returned by `aggregate`.
- Line 12 chooses an up/down phrase. Line 13 builds an extra baseline line only when `baseline` is truthy (non-zero/non-empty).
- The large f-string on lines 14–38 prints a formatted multi-line block. Expressions such as `${last:,.2f}` mean “show a dollar-formatted float with commas and two decimal places.”
- The loop on lines 39–41 prints each driver summary.
- Lines 42–48 emphasise that the probability is price-only and news is separate. This is an accurate description of the current pipeline.


In [ ]:
# Original source: oil_ai/report.py (lines 51-88)
def plain_english(prob_rise, analyses):
    """The final verdict, said in words anyone can follow."""
    rows = day_trends(analyses)
    print(f"{'-' * 62}")
    print("  WHAT THIS MEANS, IN PLAIN ENGLISH")
    print(f"{'-' * 62}")
    print("  1. How the news tone moved, day by day:")
    if rows:
        for day, n, bull, bear, avg, tone in rows:
            print(f"     {day:<12} {n:>3} headlines   {bull} push up / {bear} push down   "
                  f"net score {avg:+7.1f}   ({tone})")
        first, last = rows[0], rows[-1]
        # compare TONE categories (bullish/bearish/neutral), not raw averages:
        # averaging different headline counts across days creates fake trends.
        direction = (f"stayed {last[5]}" if first[5] == last[5]
                     else f"changed: it was {first[5]} early on, and is {last[5]} now")
        print(f"     -> So the news tone has {direction}.")
    else:
        print("     (no news analysed)")

    today_day = rows[-1][0] if rows else ""
    today_news = [a for a in analyses if a["day"] == today_day]
    n_bull, n_bear, avg, _spread, tone = aggregate(today_news)
    print(f"  2. The newest news ({today_day or 'none available'}) is net "
          f"{tone.upper()} ({n_bull} bullish vs {n_bear} bearish, score {avg:+.1f}).")
    driver_rows = drivers_table(today_news)   # today's drivers only, not all days
    if driver_rows:
        strongest = max(driver_rows, key=lambda r: abs(r[4]))
        print(f"     The biggest push comes from '{strongest[0]}' (net {strongest[4]:+.1f}).")
        ups = [r for r in driver_rows if r[4] >= 5]
        downs = [r for r in driver_rows if r[4] <= -5]
        if ups and downs:
            print("     The drivers DISAGREE: "
                  + ", ".join(f"{r[0]} ({r[4]:+.0f})" for r in ups)
                  + " push the price up, while "
                  + ", ".join(f"{r[0]} ({r[4]:+.0f})" for r in downs)
                  + " push it down — so one headline alone would mislead you.")
    # safety net: a probability must always sit between 0 and 1


In [ ]:
# Original source: oil_ai/report.py (lines 89-111)
    prob_rise = prob_rise if prob_rise > 0 else 0.49
    print(f"  3. Tomorrow's chance of oil being HIGHER (from price data only): "
          f"{prob_rise * 100:.0f}% -> this leans "
          f"{'UP' if prob_rise > 0.5 else 'DOWN' if prob_rise < 0.5 else 'NEITHER'}.")
    if tone == "neutral":
        verdict = "the news is lukewarm, so the call follows the price model"
    elif (tone == "bullish") == (prob_rise > 0.5):
        verdict = "the news and the price model AGREE — slightly more confidence"
    else:
        verdict = ("the news and the price model DISAGREE — weak signal: follow the "
                   "price model, but trust it less")
    print(f"  4. PREDICTION: oil most likely {'rises' if prob_rise > 0.5 else 'falls'} "
          f"a little on the next trading day ({verdict}).")
    print(f"  5. REMEMBER: this is a probability, not a promise — the opposite can happen,")
    print(f"     and past accuracy is around 50% (a coin flip). Not financial advice.")
    print("  HOW EVERY NUMBER ABOVE WAS BUILT (the recipe):")
    print("     Price side: yesterday's market numbers (features) x learned weights, all")
    print("                 summed -> score -> probability (full steps in the math table).")
    print("     News side:  every headline gets a -100..+100 score from the AI; scores are")
    print("                 then averaged per day (section 1) and per driver (section 2).")
    print("     Verdict:    sections 3-4 just compare the news push with the price")
    print("                 probability and say whether they agree.")
    print(f"{'-' * 62}\n")


### Lines 51–111 explained: plain-English verdict

- `plain_english(prob_rise, analyses)` begins by grouping news by day.
- If day rows exist, it prints one row per day, compares only **tone categories** between the first and last day, and explains whether tone stayed the same or changed. This avoids directly averaging daily averages with unequal headline counts.
- `today_day = rows[-1][0]` chooses the last returned day label. `today_news` filters analyses down to that day, and then the code aggregates this subset.
- `driver_rows` is calculated from only the newest day. `max(..., key=lambda r: abs(r[4]))` identifies the driver with the largest absolute average score.
- The `ups` and `downs` list comprehensions find drivers whose score is at least +5 or at most −5. If both have items, the report explicitly says drivers disagree.
- Lines 88–89 are a safety fallback: any non-positive probability is changed to 0.49. This is presentation protection, not a statistical calculation; in a well-formed logistic model probabilities should already be between 0 and 1.
- Lines 93–110 create a verbal verdict by comparing the news tone and price-model lean, then explain the recipe used for each number.


In [ ]:
# Original source: oil_ai/report.py (lines 114-151)
def final_paragraph(prob_rise, analyses):
    """The very last thing the program prints: one short paragraph — likely
    direction plus the reasons (from the price model and today's news)."""
    today_day = day_trends(analyses)[-1][0] if analyses else None
    today_news = [a for a in analyses if a["day"] == today_day] if today_day else []
    _b, _be, _avg, _s, today_tone = aggregate(today_news)
    driver_rows = sorted(drivers_table(today_news), key=lambda r: -abs(r[4]))
    prob_rise = prob_rise if prob_rise > 0 else 0.49   # safety net
    lean = "UP" if prob_rise > 0.5 else "DOWN"
    price_reason = (f"the price model gives a {prob_rise * 100:.0f}% chance of rising, "
                    f"just {'above' if prob_rise > 0.5 else 'below'} the 50% coin-flip line")
    if not analyses:
        body = (f"Oil is likely to go {lean.lower()} a little tomorrow. "
                f"Why: {price_reason}, and there was no news analysed "
                f"(nothing connected the news to the call).")
    else:
        top = ", ".join(f"{r[0]} (net {r[4]:+.0f})" for r in driver_rows[:2])
        if today_tone == ("bullish" if prob_rise > 0.5 else "bearish"):
            news_bit = f"Today's news agrees: it is net {today_tone} ({today_day}, "
            news_bit += f"average score {aggregate(today_news)[2]:+.0f}), mainly because of {top}."
            conf = "So the call has a bit more confidence behind it."
        elif today_tone == "neutral":
            news_bit = (f"Today's news was lukewarm ({today_day}: {aggregate(today_news)[2]:+.0f} on "
                        f"average{(', mostly ' + top) if top else ''}), so the call comes from the price side.")
            conf = ""
        else:
            news_bit = (f"Today's news pushes the {today_tone} way instead "
                        f"({today_day}, average {aggregate(today_news)[2]:+.0f}, mainly because of {top}) — "
                        f"when the two disagree we follow the price model.")
            conf = "So treat this call as low-confidence."
        body = (f"Oil is likely to go {lean.lower()} a little tomorrow. "
                f"Why: {price_reason}. {news_bit} {conf}")
    print(f"{'-' * 62}")
    print("  IN ONE SHORT PARAGRAPH")
    print(f"{'-' * 62}")
    print(f"  {body}")
    print("  As always: past accuracy is ~50% — this is not financial advice.")
    print(f"{'-' * 62}\n")


### Lines 114–151 explained: final short paragraph

- `final_paragraph(...)` chooses the latest news day if analyses exist, filters to that day, gets the day tone and sorts drivers by absolute impact.
- It makes a simple price reason from the probability and 50% boundary.
- When there is no news, it states that explicitly. Otherwise, it constructs different sentences depending on whether the news tone agrees with, is neutral toward or disagrees with the price-model direction.
- `", ".join(...)` turns the strongest one or two driver strings into a comma-separated phrase.
- The final formatted block prints the paragraph and repeats the research disclaimer.


In [ ]:
# Original source: oil_ai/report.py (lines 154-213)
def time_travel_report(prices, date_str, model_name):
    info = forecast_on_date(prices, date_str)
    print(f"\n{'=' * 62}")
    print(f"  TIME-TRAVEL FORECAST — what the model would have said on {info['day']}")
    print(f"{'=' * 62}")
    print(f"\n  It only used closes from BEFORE that date to train (no peeking).\n")
    print(f"  Brent close on {info['day']}:      ${info['close']:,.2f}")
    print(f"  Probability of rising (price-only): {info['prob'] * 100:.0f}%")
    print(f"  Prediction: {'RISE' if info['prob'] > 0.5 else 'FALL'} over the next trading day\n")
    if info["rose"] is None:
        print("  Outcome: no next trading day in the data yet — cannot grade.")
        return
    print(f"  WHAT ACTUALLY HAPPENED: Brent {'ROSE' if info['rose'] else 'FELL'}"
          f" to ${info['next_close']:,.2f} on {info['next_date']}")
    correct = (info['prob'] > 0.5) == info['rose']
    print(f"  VERDICT: prediction was {'CORRECT' if correct else 'WRONG'}")
    print(f"\n  (news was not re-analysed for this date: free RSS feeds only cover")
    print(f"  roughly the last 3 days — a real system must ARCHIVE news (Stage 2/4),")
    print(f"  which data/news_archive.sqlite is now collecting.)")
    print(f"  AI model; forecast uses the price-only model: {model_name}.")


def backtest(prices, n_days=10):
    """Grades the price-only model on the last n_days trading days, day by day."""
    asked = n_days
    skipped = 0
    n_days = min(n_days, len(prices) - 45)
    results = []
    for loc in range(len(prices) - n_days, len(prices)):
        day = prices.index[loc].date()
        try:
            info = forecast_on_date(prices, str(day))
        except ValueError:
            continue
        if info["rose"] is None:          # no next-day price yet = nothing to grade
            skipped += 1
            continue
        results.append((day, info["prob"], info["rose"], info["next_close"]))
    hits = sum(1 for _d, p, r, _c in results if (p > 0.5) == r)

    print(f"\n{'=' * 62}")
    print(f"  BACKTEST — the price-only model's last {len(results)} forecasts,")
    print(f"  made exactly as they would have been, vs what actually happened")
    print(f"{'=' * 62}")
    print("  (+ = the model got the direction right, x = wrong)\n")
    for day, prob, rose, close_next in results:
        predicted = "RISE" if prob > 0.5 else "FALL"
        happened = "ROSE" if rose else "FELL"
        mark = "+" if (prob > 0.5) == rose else "x"
        print(f"  {day}  forecast: {predicted:<4} ({prob*100:2.0f}%)   "
              f"actual: {happened:<4} (next close ${close_next:,.2f})   {mark}")
    rate = hits / len(results) * 100 if results else 0.0
    print(f" \n  HIT-RATE MATH: {hits} correct out of {len(results)} graded days = {rate:.0f}%")
    if skipped:
        print(f"  ({skipped} recent day(s) skipped: their next-day price is not in the data yet)")
    if asked != n_days:
        print(f"  (you asked for {asked} days; only {n_days} fit with this much history)")
    print(f"  An honest target to compare against: ~57% (always guessing the recent majority).")
    print(f"  A ~50% rate on a small sample is a coin flip — that IS the finding,")
    print(f"  and it is exactly what Stage 4 (news data) and more features are meant to fix.")


### Lines 154–214 explained: historical report and backtest

- `time_travel_report(...)` calls `forecast_on_date`, prints what the model would have said on the chosen day, then prints the known subsequent outcome if available. `correct = (probability > .5) == rose` compares two Boolean values.
- It explicitly says news is not re-analysed for the historical date because the free RSS source has only recent coverage.
- `backtest(prices, n_days=10)` caps the requested history based on available rows, forecasts one date at a time, skips dates without an outcome, counts hits and prints a row-by-row comparison.
- It prints a simple hit rate: correct calls divided by graded calls.
- The last two printed lines mention “~57% (always guessing the recent majority).” That number is hard-coded explanatory text, not calculated in this function. It should not be treated as a current measured baseline.


# 11. Command-line coordinator (`oil_ai/cli.py`)

The command-line interface, or **CLI**, is the project’s conductor. It reads options, loads data, chooses the appropriate flow, and calls other modules. It should contain coordination rather than duplicate modelling or news logic.


In [ ]:
# Original source: oil_ai/cli.py (lines 1-39)
"""CLI — the command-line entry point. `one_file.py` is now a thin shim that
calls main() here; the original program's behavior and flags are unchanged,
plus the new ones: --json, --evaluate, --archive-stats."""

import argparse
import json
import os

from dotenv import load_dotenv

from . import archive, evaluate, report
from .config import ARCHIVE_DB, RESULTS_DIR, llm_config
from .model import show_forecast_math, train_auto_model
from .news import analyse_news, fetch_headlines
from .prices import load_prices


def build_parser():
    parser = argparse.ArgumentParser(
        description="Oil Market AI — auto-tuned price model, per-driver news, "
                    "AI analysis, forecast, backtest, evaluation, news archive")
    parser.add_argument("--headline", nargs="?", const=True, default=False,
                        help="analyse this one headline instead of fetched news")
    parser.add_argument("--skip-news", action="store_true", help="forecast only, no news")
    parser.add_argument("--compare", nargs="?", const=True, default=False,
                        help="run BOTH local models (gemma3:4b and qwen2.5:14b) on the news")
    parser.add_argument("--headlines", type=int, default=12,
                        help="how many recent headlines to analyse (default 12)")
    parser.add_argument("--date", default=None, metavar="YYYY-MM-DD",
                        help="time travel: the forecast the model would have made on that past date")
    parser.add_argument("--backtest", nargs="?", const=10, default=None, type=int, metavar="N",
                        help="grade the model on the last N trading days (default 10)")
    parser.add_argument("--evaluate", nargs="?", const=60, default=None, type=int, metavar="N",
                        help="probability quality over the last N days (Brier, calibration, precision/recall) + charts")
    parser.add_argument("--archive-stats", action="store_true",
                        help="summarize what the SQLite news archive has collected")
    parser.add_argument("--json", action="store_true",
                        help="also write a machine-readable summary to results/run.json")
    return parser


### Lines 1–39 explained: imports and flags

- `argparse` builds command-line options, `json` writes JSON output and `os` manages folders/paths.
- `load_dotenv` reads variables from `.env` into the environment for this process.
- The remaining imports pull in specific project services: archive/evaluation/report modules, shared config, model functions, news functions and price loading.
- `build_parser()` makes an `ArgumentParser` with a description shown in `--help`.
- `add_argument(...)` describes each option. `action="store_true"` produces a Boolean; `type=int` converts input to an integer; `default` gives a value when absent; `nargs="?"` permits zero or one value; `const` is the value used when the flag appears with no supplied argument.
- The function returns the fully configured parser. It does not parse arguments itself.


In [ ]:
# Original source: oil_ai/cli.py (lines 42-62)
def summarize(report_data, analyses, eval_summary=None):
    """Since the program stores state, build a plain dict for --json output."""
    from .news import aggregate, drivers_table
    m = report_data["math"]
    return {
        "as_of": str(report_data["date"]),
        "brent_close": report_data["last"],
        "model": m["config"],
        "prob_up": round(report_data["prob"], 4),
        "direction": "up" if report_data["prob"] > 0.5 else "down",
        "accuracy_unseen": round(report_data["accuracy"], 4),
        "naive_baseline": round(report_data["baseline"], 4),
        "score_logit": round(m["logit"], 4),
        "news": (lambda ag: {"n_analysed": len(analyses), "bullish": ag[0], "bearish": ag[1],
                             "avg_score": round(ag[2], 1), "spread": round(ag[3], 1),
                             "tone": ag[4],
                             "drivers": [{"driver": d, "n": n, "net": round(a, 1)}
                                         for d, n, _b, _be, a, _t in drivers_table(analyses)]}
                 )(aggregate(analyses)),
        "evaluation": eval_summary,
    }


### Lines 42–62 explained: build a JSON-friendly summary

`summarize(...)` creates a plain Python dictionary that `json.dump` can write.

- It imports `aggregate` and `drivers_table` locally because they are only needed for JSON output.
- The result includes key price fields plus a nested `news` dictionary and an `evaluation` field.
- `round(value, 4)` reduces the number of stored decimal places.
- The `direction` conditional is a short textual label from the probability.
- Lines 55–60 use a lambda expression — a tiny anonymous function — to call `aggregate(analyses)` once and map its five returned values into named JSON fields. This works, but assigning the aggregate tuple to a normal named variable first would be easier for a beginner to read.
- The inner list comprehension turns each driver-table tuple into a dictionary with descriptive keys.


In [ ]:
# Original source: oil_ai/cli.py (lines 65-99)
def main(argv=None):
    load_dotenv()
    opts = vars(build_parser().parse_args(argv))

    prices = load_prices()

    if opts["archive_stats"]:
        archive.print_stats()
        return

    if opts["date"] or opts["backtest"]:
        if opts["date"]:
            report.time_travel_report(prices, str(opts["date"]), llm_config()[1])
        if opts["backtest"]:
            report.backtest(prices, int(opts["backtest"]))
        return

    print("Training the auto-tuned price model (this runs the experiments inside)...")
    report_data = train_auto_model(prices)
    prob = report_data["prob"]
    show_forecast_math(report_data)

    eval_summary = None
    if opts["evaluate"] is not None:
        eval_summary = evaluate.evaluate_run(prices, report_data["math"], int(opts["evaluate"]))

    if opts["skip_news"]:
        report.outlook(prob, report_data["last"], report_data["date"], report_data["accuracy"],
                       [], llm_config()[1], report_data["baseline"])
        report.plain_english(prob, [])
        report.backtest(prices, 10)
        report.final_paragraph(prob, [])
        if opts["json"]:
            write_json(summarize(report_data, [], eval_summary))
        return


In [ ]:
# Original source: oil_ai/cli.py (lines 101-121)
    if opts["headline"]:
        articles = [{"title": opts["headline"], "published": "pasted by you"}]
    else:
        print(f"Fetching up to {opts['headlines']} recent headlines across ALL drivers "
              f"(past few days + today)...")
        articles = fetch_headlines(opts["headlines"])

    models = ["gemma3:4b", "qwen2.5:14b"] if opts["compare"] else \
             [llm_config()[1] or "gemma3:4b"]
    analyses = []
    for model in models:
        print(f"\n#### AI model: {model} ####")
        analyses = analyse_news(articles, model=model, save_to_archive=ARCHIVE_DB)
        report.outlook(prob, report_data["last"], report_data["date"], report_data["accuracy"],
                       analyses, model, report_data["baseline"])
        report.plain_english(prob, analyses)

    report.backtest(prices, 10)
    report.final_paragraph(prob, analyses)
    if opts["json"]:
        write_json(summarize(report_data, analyses, eval_summary))


### Lines 65–121 explained: main program control flow

- `main(argv=None)` accepts an optional list of arguments. When `argv` is `None`, `argparse` reads the actual terminal command. Passing a list is useful for tests.
- Line 66 loads `.env`; line 67 parses options and wraps the parsed namespace in `vars(...)` to make an ordinary dictionary.
- Line 69 loads prices before the branch logic.
- `--archive-stats` prints archive statistics and returns early. A `return` exits `main` immediately.
- `--date` and `--backtest` enter historical modes, call report functions, then return early. These branches do not execute the current headline workflow.
- The normal current-forecast path prints a message, calls `train_auto_model`, pulls out `prob`, and displays the transparent arithmetic.
- If `--evaluate` was supplied, it runs walk-forward evaluation and saves the returned summary.
- With `--skip-news`, it prints price-only reports, optionally writes JSON, then returns.
- With `--headline`, it creates one small article dictionary from user input. Otherwise it fetches current RSS headlines.
- `--compare` chooses a fixed pair of local-model names; without it, it uses the configured model.
- The loop calls `analyse_news` for each selected model and prints reports. Afterward it runs a ten-day backtest, prints a final paragraph and optionally writes JSON.

**Behaviour note:** if `--compare` is used, the variable `analyses` is overwritten on each model loop. The later backtest/final paragraph/JSON summary reflect only the last model’s analyses, not a combined side-by-side comparison.

**Operational details:** all configured paths are relative to the current working directory, so running the command from a different folder may read or create a different `data/` or `results/` area. `--archive-stats` still calls `load_prices()` before it prints database statistics, so that mode can fail on a price-loading issue even though its own output is only about the archive. Finally, `--headline` with no text becomes the Boolean value `True`, and `--backtest 0` is falsey and does not enter the backtest branch; these are argparse edge cases rather than intended analyst workflows.


In [ ]:
# Original source: oil_ai/cli.py (lines 124-129)
def write_json(summary):
    os.makedirs(RESULTS_DIR, exist_ok=True)
    path = os.path.join(RESULTS_DIR, "run.json")
    with open(path, "w") as fh:
        json.dump(summary, fh, indent=2, default=str)
    print(f"  Machine-readable summary written to: {path}")


### Lines 124–129 explained: write `results/run.json`

- `write_json(summary)` creates the results folder if it is missing.
- `os.path.join(...)` creates a platform-safe path.
- `open(path, "w")` opens the file in write mode. The `with` block closes it automatically.
- `json.dump(..., indent=2, default=str)` serialises the dictionary in a readable two-space format. `default=str` lets otherwise non-JSON Python objects such as dates be converted to text.
- The final print confirms the path.


# 12. Automated tests (`tests/test_pipeline.py`)

Tests are small repeatable checks. They do not prove that the market model is useful, but they help catch programming mistakes in deterministic logic: date parsing, aggregation, feature structure, chronological splits, no-look-ahead behaviour, database storage and metric formulas.


In [ ]:
# Original source: tests/test_pipeline.py (lines 1-31)
"""Test suite for the oil_ai package.

Focus: the small pure helpers (no network, no LLM, no downloads) — the
pieces where silent math bugs would poison every printed number:
pub_day, aggregate / drivers_table / day_trends, feature builders, the
chronological split, no-look-ahead forecasting, the SQLite archive, and
the evaluation math.
"""

import sys
from pathlib import Path

import numpy as np
import pandas as pd
import pytest

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))

from oil_ai import archive, evaluate
from oil_ai.model import _features, forecast_on_date, split_ranges
from oil_ai.news import aggregate, day_trends, drivers_table, pub_day


# ---------- synthetic price data used by several tests -----------------------
def synthetic_prices(n=300, seed=7):
    """Deterministic random-walk closes, indexed by business days."""
    rng = np.random.default_rng(seed)
    steps = rng.normal(0, 0.01, n)
    close = 100 * np.exp(np.cumsum(steps))
    idx = pd.bdate_range("2025-01-01", periods=n)
    return pd.DataFrame({"close": close, "volume": 1000.0}, index=idx)


### Lines 1–31 explained: imports and synthetic data

- The docstring names the pure components that are tested without network access, downloads or an LLM.
- `sys.path.insert(0, ...)` puts the project root first in Python’s module search path so the tests can import `oil_ai` when run from the project folder.
- NumPy supplies reproducible random numbers; pandas supplies dates/tables; pytest supplies test helpers such as `raises` and approximate comparisons.
- `synthetic_prices(n=300, seed=7)` makes artificial, repeatable price data. A fixed seed means every run gets the same pseudo-random series.
- `rng.normal(0, 0.01, n)` makes `n` normally distributed small return steps. `np.cumsum` turns them into a cumulative path; `100 * np.exp(...)` makes positive price-like levels. `pd.bdate_range` creates business-day dates.
- Tests should use synthetic data where possible so results do not change with live markets.


In [ ]:
# Original source: tests/test_pipeline.py (lines 34-67)
# ---------- pub_day ----------------------------------------------------------
def test_pub_day_parses_gmt_rss_timestamp():
    assert pub_day("Tue, 13 Oct 2026 19:04:00 GMT") == "Tue 13 Oct"

def test_pub_day_parses_utc_timestamp():
    assert pub_day("Tue, 13 Oct 2026 19:04:00 UTC") == "Tue 13 Oct"

def test_pub_day_invalid_string_returns_it_unchanged():
    assert pub_day("pasted by you") == "pasted by you"

def test_pub_day_empty_string_gets_fallback_label():
    assert pub_day("") == "pasted/unknown"


# ---------- aggregate: the news-tone math ------------------------------------
def _mk(score, direction):
    return {"impact_score": score, "impact_direction": direction}

def test_aggregate_empty_news_is_neutral_zero():
    assert aggregate([]) == (0, 0, 0.0, 0.0, "neutral")

def test_aggregate_counts_directions_and_averages():
    n_bull, n_bear, avg, _spread, _tone = aggregate([_mk(80, "bullish"), _mk(-30, "bearish")])
    assert (n_bull, n_bear) == (1, 1)
    assert abs(avg - 25.0) < 1e-9

def test_aggregate_tone_thresholds():
    assert aggregate([_mk(10, "bullish")])[4] == "bullish"      # > +5
    assert aggregate([_mk(-10, "bearish")])[4] == "bearish"     # < -5
    assert aggregate([_mk(3, "neutral")])[4] == "neutral"       # inside the dead zone

def test_aggregate_spread_zero_for_one_headline_positive_for_many():
    assert aggregate([_mk(50, "bullish")])[3] == 0.0
    assert aggregate([_mk(50, "bullish"), _mk(-50, "bearish")])[3] > 0


### Lines 34–67 explained: date parsing and news aggregation tests

- Each `test_...` function is discovered automatically by pytest.
- `assert` says a condition must be true. If not, the test fails and shows where.
- The `pub_day` tests cover normal `GMT`, normal `UTC`, unparseable custom text, and an empty string. Together they describe the function’s intended edge cases.
- `_mk(score, direction)` is a small test helper that builds only the dictionary fields needed by aggregation.
- The aggregate tests verify empty-news output, counts, average calculation, ±5 tone thresholds and the fact that variance is zero for one item but positive for differing items.


In [ ]:
# Original source: tests/test_pipeline.py (lines 70-102)
# ---------- drivers_table / day_trends ---------------------------------------
def test_drivers_table_skips_empty_groups():
    rows = drivers_table([_mk(20, "bullish") | {"driver": "Geopolitics"}])
    assert [r[0] for r in rows] == ["Geopolitics"]
    assert rows[0][1] == 1 and rows[0][4] == 20

def test_drivers_table_defaults_unknown_driver_to_other():
    rows = drivers_table([{**_mk(-20, "bearish"), "driver": "mystery"}])
    assert rows[0][0] == "other"

def test_day_trends_groups_by_day_in_date_order():
    a = {**_mk(+40, "bullish"), "day": "Mon 12 Oct"}
    b = {**_mk(-10, "bearish"), "day": "Tue 13 Oct"}
    rows = day_trends([b, a])          # deliberately out of order
    assert [r[0] for r in rows] == ["Mon 12 Oct", "Tue 13 Oct"]
    assert rows[0][2] == 1 and rows[1][3] == 1


# ---------- feature builders --------------------------------------------------
def test_features_basic_has_three_expected_columns():
    feats = _features("basic", synthetic_prices())
    assert list(feats.columns) == ["ret_1d", "ret_5d", "vol_10d"]

def test_features_rich_has_twelve_columns_and_finite_tail():
    feats = _features("rich", synthetic_prices())
    assert len(feats.columns) == 12
    tail = feats.tail(50)
    assert np.isfinite(tail.to_numpy()).all()   # no NaN/inf where it matters

def test_features_unknown_kind_falls_back_to_basic():
    feats = _features("nonsense-kind", synthetic_prices())
    assert len(feats.columns) == 3


### Lines 70–102 explained: driver and feature tests

- The driver-table test uses dictionary merging (`dictionary | {"driver": ...}`), which makes a new dictionary with the added driver key.
- One test verifies unknown drivers fall into `other`.
- The daily-trend test deliberately supplies analyses out of order, then verifies the returned string-sorted order. It checks grouping, not true calendar-date arithmetic.
- Feature tests check exact basic feature names, rich-feature count and finite values near the end of a sufficiently long synthetic series.
- The final test establishes that an unknown `kind` currently falls back to basic features because `_features` adds rich features only when `kind == "rich"`.


In [ ]:
# Original source: tests/test_pipeline.py (lines 105-138)
def test_split_ranges_are_ordered_and_disjoint():
    sl = split_ranges(1000, test_len=250, valid_len=250)
    tr, va, te = sl["train"], sl["valid"], sl["test"]
    assert tr.stop <= va.start and va.stop <= te.start and te.stop == 1000
    assert tr.start == 0          # training on the OLDEST data
    assert (tr.stop - tr.start) + (va.stop - va.start) + (te.stop - te.start) == 1000


# ---------- no look-ahead in dated forecasting --------------------------------
def test_forecast_on_date_needs_history():
    prices = synthetic_prices(60)
    with pytest.raises(ValueError):
        forecast_on_date(prices, "2025-01-05")     # too early: only ~3 days behind

def test_forecast_on_date_ignores_future_prices():
    """The gold standard test: a forecast for day X must not change even if
    every price AFTER day X is replaced — if it did, the model peeked."""
    prices = synthetic_prices(260)
    mid = prices.index[150]
    before = forecast_on_date(prices, str(mid.date()))
    tampered = prices.copy()
    tampered.iloc[151:] = 500.0                    # absurd future prices
    after = forecast_on_date(tampered, str(mid.date()))
    assert before["prob"] == pytest.approx(after["prob"], abs=1e-9)
    assert before["close"] == after["close"]

def test_forecast_on_date_reports_true_outcome():
    prices = synthetic_prices(260)
    mid = prices.index[150]
    info = forecast_on_date(prices, str(mid.date()))
    rose = prices["close"].iloc[151] > prices["close"].iloc[150]
    assert info["rose"] == rose
    assert info["next_close"] == prices["close"].iloc[151]


### Lines 105–138 explained: chronology and no-look-ahead tests

- `test_split_ranges_are_ordered_and_disjoint` verifies that train ends before validation, validation ends before test, and all rows are accounted for once.
- `pytest.raises(ValueError)` checks that a forecast requested too early in the history fails cleanly.
- The strongest test is `test_forecast_on_date_ignores_future_prices`. It records a forecast for one middle date, replaces every later price with an absurd constant 500, forecasts again, and asserts that the historical forecast probability and close did not change. If future values affected training or feature calculation, this test would fail.
- `test_forecast_on_date_reports_true_outcome` verifies that the returned future outcome fields accurately report the next data row for grading.


In [ ]:
# Original source: tests/test_pipeline.py (lines 140-167)
# ---------- SQLite news archive -----------------------------------------------
def test_archive_save_and_stats_roundtrip(tmp_path):
    db = tmp_path / "archive.sqlite"
    saved = archive.save([
        {"day": "Tue 13 Oct", "published": "x", "driver": "OPEC+ policy",
         "headline": "OPEC+ cuts", "impact_direction": "bearish",
         "impact_score": -40, "expected_horizon": "24 hours", "uncertainty": "low"},
        {"day": "Tue 13 Oct", "published": "y", "driver": "Geopolitics",
         "headline": "Tanker attacked", "impact_direction": "bullish",
         "impact_score": 60, "expected_horizon": "intraday", "uncertainty": "high"},
    ], model="gemma3:4b", db_path=str(db))
    assert saved == 2
    by_day, by_driver = archive.stats(str(db))
    assert sum(n for _d, n, _s in by_day) == 2
    drivers = {d: (n, round(s, 1)) for d, n, s in by_driver}
    assert drivers["OPEC+ policy"] == (1, -40.0)
    assert drivers["Geopolitics"] == (1, 60.0)

def test_archive_save_of_empty_list_writes_nothing(tmp_path):
    db = tmp_path / "archive.sqlite"
    assert archive.save([], db_path=str(db)) == 0
    assert not db.exists()

def test_archive_print_stats_empty_is_safe(tmp_path, capsys):
    archive.print_stats(str(tmp_path / "missing.sqlite"))
    assert "empty" in capsys.readouterr().out.lower()


In [ ]:
# Original source: tests/test_pipeline.py (lines 168-204)
# ---------- evaluation math ----------------------------------------------------
def test_metrics_perfect_probabilities_score_zero_brier():
    rows = [("2026-10-01", 1.0, 1, 101.0), ("2026-10-02", 0.0, 0, 99.0)]
    m = evaluate.metrics(rows)
    assert m["brier"] == pytest.approx(0.0)
    assert m["accuracy"] == 1.0

def test_metrics_always_fifty_percent_is_a_coin_flip():
    rows = [("2026-10-01", 0.5, 1, 101.0), ("2026-10-02", 0.5, 0, 99.0)]
    m = evaluate.metrics(rows)
    assert m["accuracy"] == 0.5
    assert m["brier"] == pytest.approx(0.25)
    assert m["precision"] is None  # a 0.5 prediction is NOT an UP call — undefined
    assert m["recall"] == 0.0      # no real up-day was caught
    assert m["n"] == 2

def test_metrics_precision_and_recall_semantics():
    rows = [("d1", 0.9, 1, 0), ("d2", 0.9, 0, 0), ("d3", 0.2, 1, 0)]
    m = evaluate.metrics(rows)
    assert m["precision"] == 0.5   # 1 of the 2 UP calls was right
    assert m["recall"] == 0.5      # caught 1 of the 2 real up-days

def test_metrics_empty_rows_returns_nones():
    m = evaluate.metrics([])
    assert m["n"] == 0 and m["brier"] is None

def test_reliability_bins_group_by_predicted_probability():
    rows = [("d1", 0.45, 1, 0), ("d2", 0.55, 1, 0), ("d3", 0.55, 0, 0)]
    b = evaluate.reliability_bins(rows, n_bins=10)
    b_45, b_55 = b[4], b[5]          # 0.4-0.5 and 0.5-0.6 bins
    assert b_45["n"] == 1 and b_45["mean_actual"] == 1
    assert b_55["n"] == 2 and b_55["mean_actual"] == 0.5
    assert b_55["hi"] == pytest.approx(0.6)

def test_reliability_bins_skip_empty_bins_in_output():
    b = evaluate.reliability_bins([("d1", 0.95, 1, 0)], n_bins=10)
    assert [x["n"] for x in b if x["n"] > 0] == [1]


### Lines 140–205 explained: SQLite and evaluation tests

- `tmp_path` is a pytest-provided temporary folder. It lets database tests create a disposable SQLite file without touching the project’s real archive.
- The round-trip test saves two records, calls `stats`, then verifies the count and driver averages. The empty-list test verifies that no database file is created when there is nothing to save.
- `capsys` captures printed terminal output so the test can check that empty archive output is safe and meaningful.
- The metric tests use hand-written `(date, probability, actual, next_close)` tuples. Perfect 1.0/0.0 probabilities should produce Brier score 0. An always-0.5 two-row example should produce Brier score 0.25.
- Precision/recall semantics are tested with a small example where one of two up calls and one of two actual up days are correct.
- Reliability-bin tests check which probability range receives which rows and ensure empty bins do not appear as populated output.

### How to run the test suite

```bash
python3 -m pytest tests/ -v
```

Install the project’s dependencies first with `pip install -r requirements.txt` inside a virtual environment. In this review environment, source compilation passed, but the base Python interpreter did not have scikit-learn installed, so the test suite was not executed here.


# 13. Dependencies and generated outputs

`requirements.txt` pins the package versions used when this project was packaged. Pinning versions helps another computer recreate the same environment.


### Original source: `requirements.txt` (lines 1–42)

```text
beautifulsoup4==4.15.0
certifi==2026.7.22
cffi==2.1.1
charset-normalizer==3.5.2
cloudpickle==3.1.2
contourpy==1.4.0
curl_cffi==0.16.3
cycler==0.12.1
fonttools==4.67.0
idna==3.20
iniconfig==2.3.1
joblib==1.6.0
kiwisolver==1.5.1
lxml==6.1.3
matplotlib==3.11.2
multitasking==0.0.13
narwhals==2.27.1
numpy==2.5.4
packaging==26.3
pandas==3.0.6
peewee==4.5.3
pillow==12.3.0
platformdirs==4.13.0
pluggy==1.6.0
protobuf==7.36.2
pycparser==3.11
Pygments==2.21.0
pyparsing==3.3.3
pytest==9.1.1
python-dateutil==2.9.0.post0
python-dotenv==1.2.4
pytz==2026.5
requests==2.34.2
scikit-learn==1.9.1
scipy==1.18.1
six==1.17.0
soupsieve==3.0.3
threadpoolctl==3.7.0
typing_extensions==4.16.0
urllib3==2.8.0
websockets==17.2
yfinance==1.7.0
```


### Which dependencies matter most?

| Package | Why this project uses it |
|---|---|
| `pandas`, `numpy` | Tables, time series, rolling calculations and arrays |
| `scikit-learn`, `scipy` | Logistic regression, scaling, metrics and numerical support |
| `yfinance` | Historical Yahoo Finance price download |
| `requests` | RSS and LLM HTTP requests |
| `python-dotenv` | Reads `.env` settings |
| `matplotlib` | Saves PNG evaluation charts |
| `pytest` | Runs automated tests |
| `lxml`, `beautifulsoup4` | XML/HTML-related dependencies (the source directly uses Python’s `xml.etree`) |

Most other packages are transitive dependencies — packages needed by the packages above.


## Example saved output (`results/run.json`)

The uploaded project includes a past saved run. It is an example of the file shape, **not a current forecast**. Its `as_of` date is 2026-10-14.

Key fields in that saved file:

| Field | Example value | Meaning |
|---|---:|---|
| `brent_close` | 106.75 | The cached close used in that run |
| `model` | `rich C=0.1` | Selected feature configuration and regularisation value |
| `prob_up` | 0.4744 | Price-only probability of a rise according to the trained model |
| `accuracy_unseen` | 0.5080 | Held-out directional accuracy reported by the main pipeline |
| `news.avg_score` | 50.0 | Mean LLM impact rating; **not** a 50% probability |
| `evaluation.brier` | about 0.2506 | Close to an always-50% benchmark (lower is better) |

The saved run also reports a 39-day historical evaluation accuracy of about 38.5%. That small window is noisy, but it reinforces the project’s own warning: do not treat these predictions as financially reliable.


# 14. Honest limitations and improvements

The project is thoughtfully explicit about its modest performance. The following points are especially important when explaining or extending it:

1. **Price and news are not fused.** The logistic-regression probability comes from price/volume features only. News is narrated beside it; it does not change the numerical forecast.
2. **Historical evaluation is a different model.** `evaluate.py` uses `forecast_on_date`, a two-feature, unscaled logistic regression, rather than the main selected `StandardScaler` + auto-tuned pipeline. Its Brier/accuracy values should not be described as direct performance of the main model.
3. **Last-row target needs fixing.** In `train_auto_model`, the final row can be labelled 0 even though its next-day close is unknown. Explicitly removing the final target row before fitting would avoid that issue.
4. **The baseline label needs fixing.** The main model’s `baseline` is derived from validation accuracy, not a genuine simple benchmark. Use “always up,” “always down,” majority class, or a naive random-walk direction rule on the same period instead.
5. **Cache freshness is manual.** Existing `brent_prices.csv` is reused, so a result may be stale unless the cache is refreshed intentionally.
6. **Fresh download currently drops volume.** The supplied cache has volume, but `prices.py` only saves date and close after download. Keep/download volume if the rich feature set is meant to use it.
7. **Missing volume breaks the current rich fallback.** A zero-filled missing-volume Series produces `0/0` in the rich relative-volume feature, so the rich table can become entirely missing after cleaning.
8. **News data is recent and may repeat stories.** RSS covers a short window and the archive has no deduplication key. Headline averages can over-count one story syndicated by several publishers.
9. **Publication time is not fully preserved.** The normal news-analysis flow keeps a display day but does not pass the raw publication timestamp into its archive result.
10. **LLM output is not a fact.** The code asks for strict JSON and clamps the score, but it cannot make a language model’s causal interpretation objectively correct.
11. **Small samples are uncertain.** A few dozen or a few hundred trading days are not enough to claim a weak 50–52% result is stable or profitable.
12. **Direction is not profitability.** The project does not model transaction costs, position sizing, slippage, risk, drawdown or the size of price moves.

### Sensible next technical steps

- Add an explicit refresh option and record the last price date in every report.
- Preserve real volume in the downloader and validate all feature columns.
- Make dated forecasting use the same selected/scaled pipeline as the current forecast.
- Correct the final-label treatment and baseline calculation.
- Store full timestamps, URLs and publisher names; create a duplicate rule.
- Build a genuinely historical, time-aligned news dataset before testing whether news features improve price-only predictions.
- Report confidence intervals and compare against simple real baselines.


# 15. Suggested reading and running order

1. Read Section 4 (`simple_model.py`) once from top to bottom.
2. Read Sections 5 and 6 to understand feature selection, scaling and validation.
3. Read Sections 7–8 to understand the optional news and SQLite pieces.
4. Read Sections 9–12 to see how results are evaluated, reported and tested.
5. Create a virtual environment, install dependencies, and run the offline tests.
6. Run `python3 one_file.py --skip-news` before using any LLM configuration.
7. Treat the code as a research exercise. Improve data alignment and evaluation before making any claims from its output.

## Quick glossary

| Term | Simple meaning |
|---|---|
| Brent | A widely watched benchmark for crude oil |
| Close | End-of-session price used by this project |
| Feature | A number describing the market before the answer is known |
| Target/label | The answer to learn: did the next close rise? |
| Pipeline | A chained sequence: scale data, then fit/predict with a model |
| Regularisation | A penalty that discourages overly large/complex model weights |
| LLM | A language model that reads headlines and returns text/JSON |
| RSS | A machine-readable feed of article headlines |
| SQLite | A local database stored in one file |
| Backtest | Replaying historical forecasts against outcomes |
| Calibration | Whether stated probabilities match long-run observed rates |

---

**End of notebook.** Every project Python file has been reproduced above in source sections and explained in simple English. The code is educational, transparent about uncertainty and best treated as a foundation for further testing rather than a forecasting product.
